# Test domain samples — VOV và Nhà thuốc Long Châu, 20 URL/domain

Chạy các ô từ trên xuống (**Run All**). Kaggle cần bật **Internet**; chỉ dùng CPU.
Notebook nhúng sẵn **40 URL**: 20 URL của `vov.vn` và 20 URL của
`nhathuoclongchau.com.vn`. Không chọn `vov2.vov.vn` hay `tiemchunglongchau.com.vn`.
Mẫu reservoir ngẫu nhiên không trùng URL, seed 42, giữ nguyên ID/URL từ
[`links_corpus_vi_en_zh.parquet`](https://huggingface.co/datasets/AIGuruTinix/ViBioMIR/blob/71be0e1239a19dfd7a717e2bc13e3bec21bd0334/links_corpus_vi_en_zh.parquet) (1,220,598 dòng).
Nguồn có 9,784 URL VOV và 79,598 URL Nhà thuốc Long Châu.
Không cần clone repo, tải Parquet hay upload code Python.

Dùng snapshot crawler và probe hiện tại: tải trực tiếp URL mẫu, retry,
giới hạn tốc độ, tải HTML/PDF, trích xuất và kiểm tra chất lượng (QC).
Hai domain chạy song song; từng domain xử lý tuần tự 20 URL.
Kết quả sẽ khác theo thời điểm và môi trường mạng.
HTTP User-Agent là Mozilla/Chrome; Accept và Accept-Language được đặt theo trình duyệt.
User-Agent không đảm bảo hết 403; kết quả thực tế được giữ nguyên.

- **VAO_DUOC**: lấy được HTTP 2xx và không gặp challenge; xem thêm trạng thái trích xuất/QC.
- **KHONG_VAO_DUOC**: đã chạy nhưng không lấy được trang đích dùng được.
- **CHUA_CHAY**: chưa có checkpoint, không tính là lỗi.

`robots_http_status` là HTTP của robots.txt; `http_status` là HTTP của trang.
Chế độ probe tải trực tiếp trang để kiểm tra truy cập, không dùng robots để chặn bước này.
Requests và Selenium được ghi riêng; HTTP 200 vẫn cần kiểm tra nội dung.
20 mẫu mỗi domain chưa đại diện cho toàn bộ URL trên website.


Phạm vi cố định: **`vov.vn`: 20 URL; `nhathuoclongchau.com.vn`: 20 URL**.

`NETWORK_CHECK_DOMAINS` chỉ gồm hai hostname này. Mặc định `MAX_NEW = None`
để chạy đủ 40 URL, log tiến độ mỗi 5 URL.
`BROWSER_TEST_MODE = 'failed_only'` thử Selenium cho URL Requests lỗi,
trang JS, text rỗng hoặc QC lỗi. Có thể đặt `off` để chỉ thử Requests.


## 1. Cấu hình

Mặc định chạy đủ **40 URL (20/domain)** đã nhúng, với `WORKERS = 2`
và `MAX_NEW = None`. Nếu giới hạn `MAX_NEW`, lần chạy đó chưa kiểm tra đủ mẫu.
Checkpoint đã có, kể cả lỗi, được bỏ qua khi chạy lại ô crawl.
Muốn thử lại toàn bộ URL, chạy lại ô cấu hình để tạo thư mục mới.
`RESUME_RUN_DIR` chỉ tiếp tục run có cùng danh sách 40 URL và snapshot code;
run 97 hostname cũ cần tạo run mới (`RESUME_RUN_DIR = None`).
`RECOVERY_RUN_DIR` có thể trỏ tới run cũ đã giải nén để tái sử dụng raw hợp lệ
cho ID/URL trùng mẫu, đồng thời tải các URL còn lại trong run mới.


In [ ]:
TARGET_DOMAINS = ('vov.vn', 'nhathuoclongchau.com.vn')
SAMPLES_PER_DOMAIN = 20
SAMPLE_SEED = 42

USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36'

WORKERS = 2
DELAY = 1.0
LOG_EVERY = 5            # Tiến độ mỗi 5 URL; URL lỗi luôn in ngay
ATTEMPTS = 3
ROBOTS_ATTEMPTS = 3
ROBOTS_CONNECT_TIMEOUT = 15.0
ROBOTS_READ_TIMEOUT = 30.0
ROBOTS_RETRY_BUDGET = 90.0
ROBOTS_FAILURE_TTL = 30.0
NETWORK_CHECK_DOMAINS = TARGET_DOMAINS
BROWSER_TEST_MODE = 'failed_only'  # off / first_each_domain_and_failures / all
BROWSER_USER_AGENT = None  # None: dùng User-Agent thực của Chrome đang chạy
BROWSER_LOAD_TIMEOUT = 45.0
BROWSER_ATTEMPTS = 2
CHROME_BINARY = None       # None: Selenium Manager tìm hoặc tải Chrome phù hợp
PAGE_CONNECT_TIMEOUT = 20.0
PAGE_READ_TIMEOUT = 45.0
PAGE_REQUEST_BUDGET = 180.0
PAGE_ATTEMPTS = 3
MAX_NEW = None           # None: toàn bộ; 10: thêm tối đa 10 URL
RESUME_RUN_DIR = None    # Hoặc đường dẫn thư mục run đã có
RECOVERY_RUN_DIR = None  # Run cũ: tạo run mới, reuse raw 2xx và thử lại các URL lỗi

from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import uuid

if sys.version_info < (3, 11):
    raise RuntimeError('Cần Python 3.11 trở lên.')
if type(WORKERS) is not int or WORKERS < 1:
    raise ValueError('WORKERS phải là số nguyên dương.')
if type(ATTEMPTS) is not int or ATTEMPTS < 1:
    raise ValueError('ATTEMPTS phải là số nguyên dương.')
if isinstance(DELAY, bool) or not isinstance(DELAY, (int, float)) or not 0 <= DELAY < float('inf'):
    raise ValueError('DELAY phải hữu hạn và không âm.')
if type(LOG_EVERY) is not int or LOG_EVERY < 1:
    raise ValueError('LOG_EVERY phải là số nguyên dương.')
if MAX_NEW is not None and (type(MAX_NEW) is not int or MAX_NEW < 1):
    raise ValueError('MAX_NEW phải là None hoặc số nguyên dương.')
if RECOVERY_RUN_DIR and RESUME_RUN_DIR:
    raise ValueError('Recovery tạo run mới; không đặt đồng thời RESUME_RUN_DIR.')
OUTPUT_BASE = (Path('/kaggle/working') if sys.platform != 'win32' and Path('/kaggle/working').is_dir()
               else Path('/content') if sys.platform != 'win32' and Path('/content').is_dir()
               else Path.cwd() / 'runs')
RUN_DIR = (Path(RESUME_RUN_DIR).expanduser().resolve() if RESUME_RUN_DIR else
           OUTPUT_BASE / ('domain_probe_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
                          + '_' + uuid.uuid4().hex[:6]))
if RESUME_RUN_DIR and not (RUN_DIR / 'manifest.json').is_file():
    raise ValueError('RESUME_RUN_DIR phải chứa manifest.json của notebook này.')
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Output:', RUN_DIR)

DIRECT_PUBLIC_PROBE = True


## 2. Cài dependencies

Cài các package crawler cần trong môi trường hiện tại. Phiên bản thực tế
và log pip được lưu trong output. Khi resume, dùng phiên bản đã lưu.


In [ ]:
import subprocess
import importlib.metadata

packages = ['pyarrow', 'requests', 'trafilatura', 'pymupdf', 'tld', 'ipython']
if DIRECT_PUBLIC_PROBE and BROWSER_TEST_MODE != 'off':
    packages.append('selenium')
requirement_file = RUN_DIR / 'notebook_requirements.txt'
install_args = ['-r', str(requirement_file)] if requirement_file.exists() else packages
installation = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', *install_args],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    encoding='utf-8', errors='replace',
)
(RUN_DIR / 'pip_install.log').write_text(installation.stdout or '', encoding='utf-8')
if installation.returncode:
    print(installation.stdout)
    raise RuntimeError('Cài dependencies thất bại; xem pip_install.log.')
versions = {name: importlib.metadata.version(name) for name in packages}
if not requirement_file.exists():
    requirement_file.write_text(''.join(f'{p}=={v}\n' for p, v in versions.items()), encoding='utf-8')
print(json.dumps(versions, indent=2))


## 3. Danh sách mẫu và snapshot crawler

40 mẫu giữ nguyên ID/URL nguồn: 20 VOV và 20 Nhà thuốc Long Châu.
Mỗi domain có `sample_index` từ 0 đến 19. Snapshot được kiểm tra SHA-256
trước khi import; notebook kiểm tra số mẫu từng domain trước khi crawl.
Manifest ghi file nguồn, revision, hash, seed và phạm vi mẫu để đối chiếu lại.


In [ ]:
SAMPLES = [{'id': 704563,
  'url': 'https://vov.vn/suc-khoe/4-loai-rau-ban-nen-tranh-trong-mua-mua-post973112.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 706114,
  'url': 'https://vov.vn/suc-khoe/bo-gan-tam-binh-to-chuc-gay-quy-tang-benh-nhan-gan-ngay-viem-gan-the-gioi-post1035616.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 1},
 {'id': 706377,
  'url': 'https://vov.vn/suc-khoe/cach-cho-con-bu-khi-me-duong-tinh-voi-covid-19-879521.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 2},
 {'id': 706678,
  'url': 'https://vov.vn/suc-khoe/cay-thuoc-quy-khong-trong-cung-moc-tua-tua-khap-vuon-it-nguoi-biet-tan-dung-post1244137.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 3},
 {'id': 706928,
  'url': 'https://vov.vn/suc-khoe/co-nen-an-bi-lon-thuong-xuyen-post1102200.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 4},
 {'id': 707097,
  'url': 'https://vov.vn/suc-khoe/cuu-song-be-trai-nguoi-nhat-bi-chan-thuong-so-nao-khi-dang-trong-thoi-gian-cach-ly-880653.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 5},
 {'id': 707299,
  'url': 'https://vov.vn/suc-khoe/dau-mua-khi-lay-lan-the-nao-post946723.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 6},
 {'id': 707479,
  'url': 'https://vov.vn/suc-khoe/dieu-gi-xay-ra-khi-ban-ngung-uong-nuoc-ngot-post1010466.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 7},
 {'id': 708399,
  'url': 'https://vov.vn/suc-khoe/loai-cay-kho-ngot-nhu-duong-la-vi-thuoc-quy-viet-nam-co-nhieu-lai-it-duoc-chu-y-post1263715.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 8},
 {'id': 708669,
  'url': 'https://vov.vn/suc-khoe/loai-qua-sieu-giau-tinh-bot-an-vao-lam-cham-lao-hoa-lai-ngua-du-benh-post1213223.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 9},
 {'id': 708775,
  'url': 'https://vov.vn/suc-khoe/loai-rau-viet-ra-cho-la-thay-la-than-duoc-bo-xuong-ngua-ung-thu-post1129703.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 10},
 {'id': 709340,
  'url': 'https://vov.vn/suc-khoe/nam-dau-bep-suy-da-tang-sau-3-ngay-che-bien-thit-lon-post1036522.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 11},
 {'id': 709402,
  'url': 'https://vov.vn/suc-khoe/nau-nuong-thuc-pham-khong-dung-cach-lam-tang-muc-cholesterol-khi-ban-su-dung-post978352.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 12},
 {'id': 709448,
  'url': 'https://vov.vn/suc-khoe/neu-ban-bi-nhip-tim-nhanh-hay-thu-nhung-loai-do-uong-nay-post972732.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 13},
 {'id': 709673,
  'url': 'https://vov.vn/suc-khoe/nguoi-viet-chi-tien-ty-moi-nam-mua-tao-do-an-kieu-nay-thanh-dau-doc-co-the-post1155173.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 14},
 {'id': 709963,
  'url': 'https://vov.vn/suc-khoe/nhung-lam-tuong-ve-vaccine-covid-19-khien-nhieu-nguoi-khong-tiem-chung-875456.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 15},
 {'id': 710876,
  'url': 'https://vov.vn/suc-khoe/su-dung-hieu-qua-thuoc-nham-trung-dich-egfr-tki-the-he-3-cho-benh-ung-thu-phoi-post1012057.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 16},
 {'id': 711104,
  'url': 'https://vov.vn/suc-khoe/tam-dem-bang-nuoc-lanh-lien-tuc-tai-trai-nguoi-dan-ong-mat-thinh-luc-hoan-toan-post1206514.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 17},
 {'id': 712207,
  'url': 'https://vov.vn/suc-khoe/viet-nam-co-nu-hoang-cua-cac-loai-trai-cay-moi-ngay-mot-qua-bo-ngang-to-yen-post1191038.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 18},
 {'id': 713042,
  'url': 'https://vov.vn/xa-hoi/lan-dau-tien-cac-tram-y-te-tai-tphcm-co-300-loai-thuoc-post1126708.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 19},
 {'id': 213425,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/bi-di-ung-thoi-tiet-nen-boi-thuoc-gi-nhanh-khoi-53300.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 214251,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/bi-trat-khop-lung-phai-lam-sao-cach-nhan-biet-trat-khop-lung.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 1},
 {'id': 217630,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/cac-trieu-chung-khi-mac-lao-phoi-la-gi-va-benh-lao-phoi-co-nguy-hiem-khong-55060.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 2},
 {'id': 219040,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/cach-dung-chi-nha-khoa-co-can-chuan-nhat-59725.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 3},
 {'id': 219815,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/cach-lam-toi-den-bang-noi-com-dien-don-gian-68993.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 4},
 {'id': 220312,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/cach-phan-biet-tri-va-ung-thu-hau-mon.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 5},
 {'id': 225132,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/chup-mach-mau-so-hoa-xoa-nen-dsa-la-gi-dung-trong-chan-doan-benh-nao.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 6},
 {'id': 229499,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/dau-tuc-tinh-hoan-nhung-khong-sung-co-nguy-hiem-khong-67806.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 7},
 {'id': 230885,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/dieu-tri-viem-va-cap-tinh-o-tre-va-bien-chung-neu-khong-chua-tri-kip-thoi.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 8},
 {'id': 238312,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/hop-thuoc-chong-soc-gom-nhung-gi.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 9},
 {'id': 246897,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/moi-mo-xong-nen-an-gi-kieng-an-gi-dam-bao-suc-khoe-70824.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 10},
 {'id': 249470,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/nghien-ruou-nguyen-nhan-trieu-chung-va-cach-xu-tri-51141.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 11},
 {'id': 250706,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/nguoi-tre-van-co-nguy-co-cao-bi-gai-dot-song-co-42025.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 12},
 {'id': 254687,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/nhung-hau-qua-cua-benh-suy-dinh-duong-o-tre-duoi-5-tuoi-41829.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 13},
 {'id': 264708,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/tac-hai-cua-thuoc-nhuom-toc-la-gi-nhung-dieu-ban-can-luu-y-de-nhuom-toc-an-toan-66186.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 14},
 {'id': 265237,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/tai-sao-nhan-vien-van-phong-thuong-gap-van-de-dau-co-vai-gay.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 15},
 {'id': 266033,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/tap-yoga-nen-tap-luc-nao-la-tot-nhat-57374.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 16},
 {'id': 268911,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/thuoc-kalimate-la-thuoc-gi-nhung-dieu-can-biet-khi-su-dung-thuoc-kalimate-69001.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 17},
 {'id': 281452,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/xet-nghiem-hpv-o-nam-gioi-va-nhung-thong-tin-can-biet-56801.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 18},
 {'id': 282916,
  'url': 'https://nhathuoclongchau.com.vn/benh/mnemophobia.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 19}]

SOURCE_PROFILE = {'rows': 1220598,
 'id_min': 583,
 'id_max': 7466161,
 'duplicate_ids': 0,
 'hosts': 94,
 'host_counts': {'article.iiyi.com': 729,
                 'ask.39.net': 32772,
                 'ask.familydoctor.com.cn': 1,
                 'baby.39.net': 3368,
                 'bachmai.gov.vn': 2783,
                 'baidianfeng.familydoctor.com.cn': 60,
                 'baoangiang.com.vn': 10738,
                 'baocantho.com.vn': 12016,
                 'baochinhphu.vn': 2302,
                 'baodanang.vn': 14500,
                 'baogialai.com.vn': 3290,
                 'baohaiphong.vn': 16286,
                 'baolangson.vn': 1981,
                 'baonghean.vn': 15941,
                 'baophutho.vn': 1288,
                 'baoquangninh.vn': 2912,
                 'baoquangtri.vn': 5483,
                 'baotayninh.vn': 1198,
                 'baothanhhoa.vn': 782,
                 'baovinhlong.com.vn': 5113,
                 'benhviennhitrunguong.gov.vn': 1200,
                 'benhvienvietduc.org': 425,
                 'bingli.iiyi.com': 5,
                 'byby.39.net': 37,
                 'cancer.39.net': 736,
                 'care.39.net': 1282,
                 'dantri.com.vn': 2063,
                 'ek.39.net': 448,
                 'fitness.39.net': 185,
                 'fk.39.net': 928,
                 'food.39.net': 594,
                 'gan.39.net': 294,
                 'gc.39.net': 59,
                 'giadinhonline.vn': 16729,
                 'gk.39.net': 90,
                 'hanoimoi.vn': 1283,
                 'health.people.com.cn': 50,
                 'heart.39.net': 1055,
                 'hellobacsi.com': 11747,
                 'jb39.com': 527,
                 'jbk.39.net': 2033,
                 'khoahocphothong.vn': 8143,
                 'laodong.vn': 31282,
                 'man.39.net': 382,
                 'medlatec.vn': 24762,
                 'mega.vietnamplus.vn': 2,
                 'nhandan.vn': 5225,
                 'nhathuoclongchau.com.vn': 79598,
                 'nk.39.net': 762,
                 'pf.39.net': 85,
                 'phunusuckhoe.giadinhonline.vn': 27577,
                 'phuyen.baodaklak.vn': 6252,
                 'pmc-ecm-healthblog.beta.pharmacity.io': 5333,
                 'pubmed.ncbi.nlm.nih.gov': 240000,
                 'qihuangzhishu.com': 29,
                 'qy.familydoctor.com.cn': 2,
                 'shen.39.net': 325,
                 'sj.39.net': 197,
                 'special.vietnamplus.vn': 1,
                 'suckhoecongdongonline.vn': 154503,
                 'suckhoedoisong.vn': 85823,
                 'suckhoeviet.org.vn': 5627,
                 'tamanhhospital.vn': 4261,
                 'test.pmphai.com': 2989,
                 'thanhnien.vn': 72111,
                 'thaythuocvietnam.vn': 2927,
                 'tiemchunglongchau.com.vn': 13281,
                 'tienphong.vn': 3021,
                 'tnb.39.net': 717,
                 'tuoitre.vn': 3893,
                 'vietnamnet.vn': 15543,
                 'vnexpress.net': 2055,
                 'vov.vn': 9784,
                 'vov2.vov.vn': 6899,
                 'wei.39.net': 225,
                 'woman.39.net': 687,
                 'www.120ask.com': 98160,
                 'www.a-hospital.com': 4701,
                 'www.baidu.com': 11,
                 'www.cnkang.com': 40046,
                 'www.familydoctor.cn': 2,
                 'www.familydoctor.com.cn': 26819,
                 'www.msdmanuals.cn': 905,
                 'www.pharmacity.vn': 69,
                 'www.qdnd.vn': 3700,
                 'www.sggp.org.vn': 1853,
                 'www.vietnamplus.vn': 3695,
                 'www.vinmec.com': 26129,
                 'www.wujue.com': 708,
                 'www.youlai.cn': 4139,
                 'www.zydcd.com': 204,
                 'youmed.vn': 7189,
                 'zhongyibaodian.net': 6061,
                 'zysjonline.com': 6591},
 'first_1000_host_counts': {'bachmai.gov.vn': 999, 'special.vietnamplus.vn': 1},
 'sample_host_counts': {'article.iiyi.com': 0,
                        'ask.39.net': 0,
                        'ask.familydoctor.com.cn': 0,
                        'baby.39.net': 0,
                        'bachmai.gov.vn': 0,
                        'baidianfeng.familydoctor.com.cn': 0,
                        'baoangiang.com.vn': 0,
                        'baocantho.com.vn': 0,
                        'baochinhphu.vn': 0,
                        'baodanang.vn': 0,
                        'baogialai.com.vn': 0,
                        'baohaiphong.vn': 0,
                        'baolangson.vn': 0,
                        'baonghean.vn': 0,
                        'baophutho.vn': 0,
                        'baoquangninh.vn': 0,
                        'baoquangtri.vn': 0,
                        'baotayninh.vn': 0,
                        'baothanhhoa.vn': 0,
                        'baovinhlong.com.vn': 0,
                        'benhviennhitrunguong.gov.vn': 0,
                        'benhvienvietduc.org': 0,
                        'bingli.iiyi.com': 0,
                        'byby.39.net': 0,
                        'cancer.39.net': 0,
                        'care.39.net': 0,
                        'dantri.com.vn': 0,
                        'ek.39.net': 0,
                        'fitness.39.net': 0,
                        'fk.39.net': 0,
                        'food.39.net': 0,
                        'gan.39.net': 0,
                        'gc.39.net': 0,
                        'giadinhonline.vn': 0,
                        'gk.39.net': 0,
                        'hanoimoi.vn': 0,
                        'health.people.com.cn': 0,
                        'heart.39.net': 0,
                        'hellobacsi.com': 0,
                        'jb39.com': 0,
                        'jbk.39.net': 0,
                        'khoahocphothong.vn': 0,
                        'laodong.vn': 0,
                        'man.39.net': 0,
                        'medlatec.vn': 0,
                        'mega.vietnamplus.vn': 0,
                        'nhandan.vn': 0,
                        'nhathuoclongchau.com.vn': 20,
                        'nk.39.net': 0,
                        'pf.39.net': 0,
                        'phunusuckhoe.giadinhonline.vn': 0,
                        'phuyen.baodaklak.vn': 0,
                        'pmc-ecm-healthblog.beta.pharmacity.io': 0,
                        'pubmed.ncbi.nlm.nih.gov': 0,
                        'qihuangzhishu.com': 0,
                        'qy.familydoctor.com.cn': 0,
                        'shen.39.net': 0,
                        'sj.39.net': 0,
                        'special.vietnamplus.vn': 0,
                        'suckhoecongdongonline.vn': 0,
                        'suckhoedoisong.vn': 0,
                        'suckhoeviet.org.vn': 0,
                        'tamanhhospital.vn': 0,
                        'test.pmphai.com': 0,
                        'thanhnien.vn': 0,
                        'thaythuocvietnam.vn': 0,
                        'tiemchunglongchau.com.vn': 0,
                        'tienphong.vn': 0,
                        'tnb.39.net': 0,
                        'tuoitre.vn': 0,
                        'vietnamnet.vn': 0,
                        'vnexpress.net': 0,
                        'vov.vn': 20,
                        'vov2.vov.vn': 0,
                        'wei.39.net': 0,
                        'woman.39.net': 0,
                        'www.120ask.com': 0,
                        'www.a-hospital.com': 0,
                        'www.baidu.com': 0,
                        'www.cnkang.com': 0,
                        'www.familydoctor.cn': 0,
                        'www.familydoctor.com.cn': 0,
                        'www.msdmanuals.cn': 0,
                        'www.pharmacity.vn': 0,
                        'www.qdnd.vn': 0,
                        'www.sggp.org.vn': 0,
                        'www.vietnamplus.vn': 0,
                        'www.vinmec.com': 0,
                        'www.wujue.com': 0,
                        'www.youlai.cn': 0,
                        'www.zydcd.com': 0,
                        'youmed.vn': 0,
                        'zhongyibaodian.net': 0,
                        'zysjonline.com': 0}}

BASELINE_DOMAINS = {'vov.vn': {'error_type': 'not_tested',
            'requested_additional': 0,
            'selected_additional': 0,
            'shortfall': 0},
 'nhathuoclongchau.com.vn': {'error_type': 'not_tested',
                             'requested_additional': 0,
                             'selected_additional': 0,
                             'shortfall': 0}}

EXPECTED_URL_COUNT = 40
EXPECTED_DOMAIN_COUNT = 2
BASELINE_ARCHIVE = ''
BASELINE_ARCHIVE_SHA256 = ''
SOURCE_SHA256 = '87e44a792d7a99fb0fde5dff7f09f1a38a0a1fdf02ddd1ac93a005736e00b0a7'
SOURCE_FILE = 'links_corpus_vi_en_zh.parquet'
SOURCE_REVISION = '71be0e1239a19dfd7a717e2bc13e3bec21bd0334'
SOURCE_URL = 'https://huggingface.co/datasets/AIGuruTinix/ViBioMIR/blob/71be0e1239a19dfd7a717e2bc13e3bec21bd0334/links_corpus_vi_en_zh.parquet'
CRAWLER_SHA256 = '34cdb323c1697857d761e3fef7d8c91f11dfd4d1fa17788b094dcb9b66b4b145'
CRAWLER_PAYLOAD = 'c-qZ<+mal|b>RDaMG*{bb%Q-MJu?7B4$zb!5RwoiK^g?*bqw0&n(pf9;&gX4tE%Us!92(c`J%Npto0@2T{}!#>##$zOxf0YEke6`0E7R52!W)Y_80ctGApwxtEUH|c1>G@>Z;6>x0C0dCof&P^tI$mN&4FDyWSv+jwb8rY;xkwlSx_x-ksEYXAx&7-Xz+KCwa%4rbGI-$f9JH%=W!B^OD&xK8EMu(xpplqb!|zVK`b8i!2U9FPYBMtni}QEG?oUNoV=mntGS*&!a4lxy~S+OyU8RP?c_`i&+t89d8(qqQ#^bCWAs1o(&dR7SD=+Thevzjk7o!zLBPrTgUNWQKV2k%ELiAoloK-9_so<d|XVDJylQMOru$}A7@-@7!`4mOk-6>{f6Pg->;`L>3Tekl1Z>Al8NpH4GhbNMH;GlYT)}{Pv+`*9OYx^hWg3>z>8oS7ZFs9^p|-w9Kt|z_2TV3ovG((RE)VVb9gd_^#*>B=Pa5H)2aTYyOWQx>-twZIEV}V3l}Xfl7VhEKhci`%{8pNeqju9F-r!3EHnh_*&>_ZxM*fn_u$>zX)^2JLq4A*g;g|5_tJvOXR5+oeD&pI62C#OH7?ChqAW{~V5wmSDlY?E{6!Jd$3gWYzy}mB=;J%(<iNYUC`Xz_qht~RrpaK6$xwA^KQ6-2WVp8WwOe=Jy#4weujl!PpZC|^x`9uxrC(1blW5~=(Dhos30oyS%Dp@Hyv=UVz2?D-=dWDzj-S8M_HN7pLw_^gdnGA0u5N!m*nZw?z4Fz2uf5vwCdonUeI*_oq;2o!7}}3FHn#x){-5_|G>WoBs`zVnU;onU_re=rxdn}s?fYwMYp{#FMKNf#ch)@kFN<L~GmX^2EIn!|yar?*w1Xs1M`;GQ)oM%iVX_a?(!xcxp$HlW`8e9T`aBm1#__RKQ1xLF;}FZX5KjffQBsTvQTb(%&f{6j&-VN_F3N}tIocv%(!i+W`P{beYBws<X)*}+PKr3ERyv+B27Cc?RdzwYMeIN#Gd&n4S&M(<y?faLXaLZ>ya>|+`pH#_csd6_R1-%P&Z22dAOn2zp7Tb2aH_XWfT)uU0cNXx78KLDUxEpavIM9H_n7)xe#+wcBpSrhR~{i3N17nxOZVQE;W-A6<$~d2I?r2B4#qOekz7alAW3>(jwU&b56Swh=xw!wc!oQ<<u8iS`U`$rH91M6VcDXM57Yn_$XmwB;<!U3vn>J%Q_!md#flT@C2Y^Uc?`ZEAnj-wW64}2X~VSp_h-Hbxa=XfVz^DC8mE9aK!EJ=v=o7&PXQL96$Ow!VB91<iZhLEGu$+j<m+(@$0~sE5pcsGi$Ph=zw_mreuv8lNVUlR{R}$ap&?jr)jJb<76<uaua)_`_Y3d-qT7wS{pYZtC(zsGpmgc`v(xR)83WXf@$4Q4d66aaR-2~p7^e^61Uwnkfp%M!k@3%=m#|e@z~I&TrHexCFnk-pZ|+Jm@>=+D!}EdnCL`Fj0j?;Fl{Q$e$QAWFN`QBKEt)LiTR^<CmjC*q0MhP76GW{OBn@8VDL~g?gbQRgG^vAen!=CBo5e@o`Z_ck$mHeeA{)e3|M-;<f)VcNcY-&VQy}+gncrau^HGow#vliT%v2CJa_^a*x9cZEoKL_SzmL5DF$m~GP@K%;90+WffY<>7N2>~9l*E%^3v0Kj@X}ehbvy^rDF&9r(7a>_64%hfWdRut8b|SJF`eaV$jLD0LGhN1_Vk!Ks%Ve;-f#0iYNSvxm@Os~kf=a7!x7B~0X+jL20>vqC|Xda9e~g^KLNCXNtggV$9Kzy@LH1RB2Nw%fLjA#OKM?mzpVRBEn(M1Hf-_hQIY{IZut1D_txE4y=YO4Qy`KjUXpwHVl+wy3D7q}>v@&}GlAx>d55qektKQ~803k!7sGe>5f_6oQfzJws_aGZ26qE298lgYq!&P(%GskeHwKN>a4Db?6%M#)s|zcf!emVn<XB~SnQ_FJ1m*@ru^H@qfY8B(Z7XboT|7l_u#bOxTom*C3p*PdyT83b8^0uE=6|*6ANjXuG;%fNFk?I3Db#)@JCmtb|A7n%vOkitY@*1(1~j$Z-reeJN@&*Lb=l>Leeb!Rw^<!zzL>VmMGOwHbFlf_-gS_by50I9%;I&}0FyT|kXP>|f?tCS6IhmkW|rZno}UO62@k6%oP5^1_0A%itRqtZN;F9ar0B3*lf}3#vSe>T?+^#DT+UxdK#GtXF<!m1038V?7WJF}#ms~FB!_L8!Pb`Ew>w^!TRj>l*bf*NRkJ2ckIF_qSvmNBs#3!=daI}MdUf51qO97U`p<@`mnVa*-V==^xJ&;C;f>>=Vp1UBAEvO%I3=Y9*us4xggcIBby~1&ceblDG5`UrqqZgJWvwM!vt#euP3vQ2n_YaD;9#2f$Q(3UV(@BGR8gzcB$+12XsubVE;Ev21cxg}73!;opMz))g7~m?iu8X+PZg219goP6&n?hbd^od51eO8^Mv%DxhXUjj5di+T56s6HeY;*~B-I*wta`mm<@x~sBK>A;;;r5DWXH8KW)3*>C^hP-*SKGblr*0m4>oLX5>5Apk(cdwnQ%~sUPM`elsiQg0VMyZC^Eu%=ua{X={q3E9Z<gjlB+;ESKqbhM75Bk(Q9P3Eowo%Sw_;{1fd}r0gp?860NT%!*zy+`&@nkq*i=9m@M+-Fm7w%VdUv{kpp2%WkY#9VPT|6z5rGWTM{^AON67BdVYr?!obzb>po9byKil?vYF~?onF42nM+H8e5Tm&S-DoKgUi>;*eu)>&29swb1~GHNgB%lS4b>9H6Wc4(MJ_h<hJ#7;z6ex{8?adUBwqpd2Bmgw?fXDP@mZqg2fyKnU<oCFglr=zA+cvp|%pzSJiY`O%uVLfsoif!Bu0J%%w`TR23#H2wCLfN>Jxi_ua>-f#n0rtxRAba85&>G294KT3!%4;lYwM<#Ti_pQB^@bJ>O#F@ZE-V_m39Y#E+$oiwwhw<2V~H-0bxu@Hg+N@$lfrIU`DqDmQBCI)uaG+nk*WKGln_7)&d7!^p@(6EE(5A&7B_{P6+`zwoVaS!aY#~aGVi#rQUougzQOAOL%zR2+%aVUI6hF#wnA+HyF0x3qeFl_2PogBu%^g4P$#rm>i*my)qGJ)YVfE<l2!BVLW;yXzS@ej|BQMlrn(<IpJprV}sO$ZfrmZ+_wo1rFR^evrZzd_P~Us~#*S4<fM3j~p?{dbPlwH1unS%%am0>C^}C_kyNVYZkl+Fw#0(6bMdbdg({t1jO*>}K!^>G5vgl77%&s%d~t(9#LfZ`HsC_5-j5HVZtDE}cSECYEP3)=Ms>1z<RI_);mDfws7aZ8}PgK+C0V381vA(770Q0I9O>4T|*YBdgr%EB#tQ6Y}qp$>8?O8>lo+lbm#`z;EaSn()9@86AaaltPOoFwd=S*Sp@cN}ytfXj;&A>!Rqj?NVw0l0ENgG`fj?t<GKwmX98J6@p=m#%^>Iw4eraUpAVxol=$9Nf89`H?ujx^1fTxYMMD50q%&#xnt>g06hjfQWg!k^^R_uJTYKSK&yw}FW--n8PEad=Jk2wxfYfO;Lj_9L<NiiRJ2}pG>P^@RNbHw2oY4XUJ7UoEG`U$H~$Jt#WgNyRkoA_^3{L_GCx7ugNxh|sOOECsX&N&We;h&LNbe*gXJzu2Y^1rS9-)%>VWHP%AgLo?a}_FR?gKg?1|DZ%*|Yubv1#6t;P>Eg<Z^qNjV4R%kY`s^ADl9b1;$>rWyIPi4imj3D84n8||8B@M;9yW-M#Cwl<hVdG6gr_bTs|TgQWVjs^iCjA%6tfq^AO7=kJ^8Ie2;!~%Vv1$Y;GU=p@OY%mv60Th7gq5Qa*qiz{!cnkqaBbO|D^kp<_#aUw?U^Li)Np#ZN3_wN#z6AaUz}4-JcNy*TpgQECev9m>QzsTu4EFU5CN)GK69DdQUPX}dk<|qGg08J*IAFZaPQty#5OBWtV%GpNf?=ZZzbGaIqdH?W9D1TL$+POd;-a{OGU{@~HlWJDp{9OB&QSR(x)_Tym5-*XfdsGRg5OqBQNk8iXCEy4m773Ut4tSQu-ZeD_&m*%0;D5lKsIT%*&2(Eq+C~kA^tnM)p7EU@;~2=9JocDA6A8D;!_FCe7icN65UKl<=a&u8R%I8GG3lp!WpQh9ghqRqO(>O0qt7T++i;^Y{gU^tX5>~5p!|<a|;$7DOMU4ZCwl_3cx=xLV6%2=^OV5q?_(aX5~{D#+gPmhDI`y!nh_(lT`v#Yc*hkYf;^>ti`gv7_*NSDOd4&NxO~xMMM&7>D7G|A}Cswo`6yKAeXjhAosuqv3p_`;8(cI&N{~boMQtMLXw?O*;<Ry#OfElK_{J}_Sj|!xF*y-i}%q~eL`jt1Fy(ZU5>RvWt1zC>w+d#h^@9o7+jqeqxl@f3}AuOy3w3y4<JL58dMI36f~;hSFpY;6G#a%m2wP6@TRoWKBo@E4MamQOeuls)PD=q&Gj4mAfCa7ed`8v;Kl$MHz~@niW}o%I_ZEML&gLEu`wTxuD#Ri1}}ClZ(OFQtNvNL8o^L+n`h;V6mByB6>*#n^*kw;RLgeeaEK9Gm`m5RIgMO`{fe<L5RY{c)cMI-<hM(&B$YN@+wuxzrL6L54&uX6T0;gR0EV37C+s#MvE9`-il9+FBl*F50T}v~E8FdhfQHdPWh84u+>Hp;Xzp2xtvT_ERJpg10>y_I*rtShh;_pk?CJApZ8ktKJVaL@NU3x1g%{$cQ4|^pFwtg;m_j`dBeas)C~Z~dvSSSws<vQ&^%d&%)S|Gg@ItrU;0{!=U_mku<*|j`*}c-=?e;NpC)}T;d(otcNShP6nB~zZR&e}I6%&jtD|;$Tk$o`LAcHy?<&~(&%Xpci<WQdD7em-xVEc)C6)Og91`Ef_PslBLD#D6gpGb(q9CV*qF*#ug+Y_gnT^XqxO2>-)YD%YxF+jI)QuR<tZ1&_~tsok_vjBWCXh}SgMX|oCp`#75TaMjZr%4H>Dbz?I6a)z3xrh#{G6&>_SSNvk=zf=1!ecfCYU4NpCJXf7C<49;-ZjHQj6E~91CVLENvr}hC4Dt{6d5mYp2%k5=zJAB%_-D&Jlt*6#!J0rGqDP;UTiM*PSg|(Km>x&yII8+L0!lvaXfEf2P+?|^Q7H4$W|+zcPP2MRZ2ldJ>mX;p#xSTw?)P=gAEoKj60<uTx(jXht3LQwNiE5BHU**^A3|w!<>L2K5n5QUxd-JgB#u!JJ_@kT&eD`OK3?&$rf$GU?Lo9OIDyzFhv|q#r}f+`y*}o$yEp$IB?VixF>BeUd)iF6kyVykf{~6jI-p3Vv(S<I28#fd#T0r)q4Kw)$Qlo&8x$WJjanW1gqUx78Oe>{?annY4f_Lz!lLWg-*E<gp)!vOfimT;Lk#6XI^P>qzwnAMt^x{DjP>(EcEZnRoWYkG^AG5peOB%j;oH(&E4k3RR*gi6TXZBF6;fTZ6kUjeQQK(7^Si{i3r?LMrDnkTp+~Q#ZD<&!`4|0DE>aU32vv`P;$n4-OC9XPNlYh#Rv)&w#jqq?v0|>!jw#t4gG^?(rsJ;F^r)^d1dRxj(4@Yy#za$x)>u>S^(MY@1icaeq#i?6Q{Ym1WBCiV#%n~Pf@)^$8Lrnlg-djua~!WZg-Y;$$5T3x)|UyO!9Qz(lP;kUebAE0IhNa9q%64yyWq9AT%%&NA$COe&10AapSTf!0e*%(s%G{2ntrCZD3Wbsk!#WnjMA)Zi$IYQgC%cqq9rE>bQEjWCN7b?d${zq#0PUVQ1QEKNw8de4y;SS+bwZ%wQ4xikcNVH|Qe<QEzaipg1o4HA^5OJ+x6I(j>8`OF!3Wpee_Mni$yYDIku6{QXnz%-PPV#=39@MzAwqg>$#tyE~iDm!21J1wa@(MfaZiTpP4^e$YWddbse0nyq@(RbSBME01ogpRY(GltdB07egpJ{Uy?XhKnIF@SEg|(sVOV!IS{29OEW{KC|kiEQmlcXDGy)_5_9mMG2Cnpi0{$XHbyUX2)$ie$LyhE1DPMS8AS>@q`A@^Y5}Li~3oY5@np>0R(OJd=m@8Wul}xYLwdQ9#P<W2WzhG?N%0~U)t{1nwqdD)*9?QowxjQ3|4rSVSSMC6Y@iVI-Z=01qVSe6LTN<n>*<F#SjPN*tUHFWDq$26WDRJd<0HeE)jvxS70~-bzUMxlb{Lh6YNvjB+^=m(-=I#PlVIlPYBeQQ}ETC3aUMH*3g7n6}@fxO3I?<-3(<f-lx#Yb9JzD`CyaS!@I;ZcIeS>cDJs0m+7C*xjgFJ|CCbVFnt>q6D{O6HYfuy&7SATW=bS$mN#$#)OI%F7kX;aOh<jslA3Pvmv!XsC^q@+6{kwewgRqqda&aivOfg8A&9i9A#r~or8-dYwDA3Dm!zfQ4%Q3lw2I^Q6~$hUlo*Ix6rS2=D-H{>O2R36X2eB%iP(Rn{E<TZH{cZ*Yc~`#Wb#q6J&eooPZf7M3!_unoM&Nn3fu84*#ElU*q(L1N;yk0rUBy%D6&Mym2kAi6tT3|ZJ&cbW;6t`|BT{NFd`2eK*voYqlk*yO^*6EyxUm~S(*1>TTln&fveqazauaKSV+$A9=Thw0C>vL%o&F5<yx3qn{N{TD$@W`+Z^K$a!{9N5QESss=WY0=TL}=z+tTTQmUzjTiKR6PNt`6mbFu*LzOU_wTt4hfH7WCu#T*DueSQDsB7C@kWjX}Tli-i|9l?*yueXa7KM3LZO$c%LdhQGm~lzO1=el}r@ut|cSWi-iyF<5?+yCAE!&|ouY@LH{T$Vg_!|$M0ujuj#GJ`wHbj>hCk1icC|ayM0!eg^ju-lki{)QOmzS@O0fMXzp|AQp1D##G{tU!a5mq&nTDDh|JYdLFxKp&Vhg!dgv7|P+c%s@v7dW7Rm#SzD(rMZarwULchfY+^p0~8nwX7bWcW;$vCXtWljdV{>09K%(<AVdgDt3<U6>yqucH29@z9H;7wn)7nuz{~M+FKgecf<jbrwr=Mzu49Rn2FdYMV9{Ry?bwX8e01z`_mBot$~5X9;``(;#Vts?fGu^|K)H?WL|x|Y+^A7$C_nbw`tICoT|-*b)g33Y*lWrv|{+E!r$zG7x>UIrmQU?<W1_cJ8Fi2w6q*8rd8cLuvERt(A(#uP^%JhL$EbyEs{Wx!ke!$(DNBFGQ(CQ%xZVrD@w#jy3HtM<&}5oXuIp1{c|Z;1?TJHmYbR^TTOD?murvy^13y=f%MdOKz%<sjS0&cQ-jfB?HmYN8ZISNm!rSzu3^sU^_*6mofURk#i6d{p=C~WDP%AAUIfWVMtCrO%e#EpYpK4igCz;fNpi3gk{w_I5%ektTUXmnPG<r3E4rSa%EN48H!dXDM`3+pS=^gqA>L5KWf?3typ&npz~x`a9%b(;)ejtuLs!7pHNBwGiCt9`G&c406R1VDV$y3=O2l-F%{OZcg9z+Y;W#P8aaxUh()La1npgmh2osV>xL6rsVun36Mm&k38rkdVTw-d@;4D1+n8gLC^9G|;Q4&jxNToB`jvlcC_G+ecWqA(;!5uRxjjz=vZdjVz?U+xu1M#{e5|FlyGH-Jb=WNt=(H3s8!EeDZcNtK>45KMM2Qu(Ax6D{uVXx-`?Bc{+oFlZX%}EyXhSMk&!lovmfv(MUY?E6>qG(X<c&SH#{wkEJIf&DkZHFJo3_NvH64gQ8gHYaOY{1TMUqm2wpcqPeYKGl1o^g7^FeVAZ#X};kCMy=WnI!*9i9+qQ&j!H^Ul1h{C?S+&i+4q*+`<*YsNdr>Hb#^(6*JmXk84##KPUK0#1>7-$@rlJ@|dO2YUHeFZG&hQk~y$tqy%$i1Zmc*lAOX}NVl>`EjuB5x+$z1(o3^3+G}0o1Ul_esiUmx_|~dq4u`5BDZmF+r6TF5zTkcpA1RfVbt5CDQMtyFq>f%8&5C&Pj3`Gut!4(ASO%u@?U)lkR1RB)rN`xk$^tlD6ly3c7Q94m#T}i3ndnmGYBtL}Mj8Su$l69;JY#j{AoI28V`q01s$2LgZrC<TvdF8=AXsMD{6=x@216FE#;&3%eb**yHs97O^W$38Ro3W3UE(~wxc6+CqfB*mRS#LC!|SP>UE=Ce9xq1A_SzClx2Z(->&Mfo(I6}6*-Nee%D(Q<s<x|Im4T5}WvW?=Y9^C_pAZ9o5Gs0X>vIsiNXbEu2Ww59Katu=g^~_51w=KNpsVILwo+ltQy2eiQS=l3xx(7B8us;7M*Xb!Y6>zkd%JYF3<#qT1E>@w462VHJuG;}=)5O8trxRk6*2AU^-pvB7ZAlD$ng1#MZIDO6xrWFMbgS41>EVtFs&6*(p=U<MH6iQ8kfGVR1&3F=CO6ShRR4+;#;4YSNB)<<T_lkz^(SlH5=u{I;0{_Mr`*jeYNegny&Mi`<L{w@27Js(L9&Qm5@kS{GmwUC%1vFW+tRNG7JYNE~OBzv}&dW5`L_3Hdm~vM0L;>@O2v4bSh~)C&032D#az_e|`m!S{<?>PZQ4U9v#KlL-`pp)yACr3{?UMBL<s7GML2ckXURQ0qGdWi;NNuQ#NVABsfi03RDD*PRIFljA@u05(*@ArfHM@ZqeUu`g?``UZua!<KIGj_ygtLCy1DgCHBus@$V2Ukp<Xqho;E)tT#EHVm8Gojys1_YKM4!76hjWndtyQuX(Rya={qqeVAYykx>C@Ckp8@Zce7T)|~DS+YbOL=lpY>y(`akMEk=2RHb$;(HO>cK{480`2do37Bm;d$%HmoGRi<H#-~|od_JUIho(G#c^69XvP!Vssas%7+spiFU0hhT6UtnsdLnQTFg?p^=|b%_JH9@_1kE!7V%O(or2;ilL2m;*E$=vsVzpbAbkA);WC=5ZEFH7=!1LRUvv>Y<t(7BEThUDLY9yGIsA~qIlih+{a}1WuaK6}6M;IyEw*^Nc;&NBtMPDh#tSMR}*z4z<JV6Lw2U4q|q<|1|mVO}o6mt||&K=zOdQ%Gk+{V~ca6G3_myM0fyI)L(y-SMnLzGBysbA(D!vhbhE+s74tfhxWi^Q586Qydv**MiwpqnbdCmXNRII1nF7^!m_5|_H~$s{W_0IlJ%4x*Hzrt{QLqF!LeLm(TFw)4*Zqjyid55N22H)s3z^UFK%zt;WX>1Jo^Y{U5X%2~Vhg=g*$+g2`L=_&y}x1*+NP|g1hgkw%&+yrVDLuUNX4L3eF-T2&ZK@wU*3p*$HIZ^3m>o&T{dW6V*>OACbJOwe*qhs#bl+G{^5@9|%;1EX%idMNKz%&9TA@MWWXz-cHszNqN`9D9=iTr^};cpWa=@1paw)fJA&c_bv1QO+-lM1x9!#E$nc8q42<fwj#c|}5A<IohL6tmi2w74$GCG>&M>|&B=*1X%MO@sFvU?Xmwb<X_}hLFJ|UdM!I`!I#;vb!iiLz=2nULo6Rs}m>pX$j1F6~i}O69KIg2))&KLu`z%>|)Cx{$QHi8FZ(0$-C4k$<%s2$K=*uB-mg8gs=3Mz`}FCB2a^y=b*8*=qVc`dVr1Xei6>DkE(5!GmL2!TGR#6t!5`JO#`UQ=Jr=PvOzwl{46xN_io?uUcLS3+jqR*c<aU;{{q2_h?P7oh?X?%w&jewX)h1W(=da4Xh?LS4@#an0L;&c!c(f4mMNV;Q)mRFT?GYr`?Uj;&q#5V<UbuysvSOhsWv;Y6w~~u_)59YrR2tM{D&K!tR2}Yj6DuLNHSBvTyc|&;%T1}M&$tC6UW}%>PGnihB+eVJ~}!Ij5o%9B_jphQqsV>AOm3zs?w#tU$e~Wy7sax1Nn3~yax&J53p)w(?qlUh*{!i*3K-)s-dGX;Y@mTRqo{CQ08<XMOKQ227<@iE5o4@IHp})oTPS(a?8m3jdcapGsbycLkgg11`qwF6+M4cL{^p{4J_DKH!Y+QD5o3?wqr_jl%}5v&93SK9Jx2U3;`N-_1vdkaNb*<Dee}|s#-68L-c7=;%9sZBuU-|hA&8l!CNJKC&Eo19+DC!dAU^gLeP6Nl{!nkbahec$yn6684~=A;Vr32A*rf0Oju2$LV>AqecR0g#IAEXu#^Dq`p?P^2hT#FHyVyOu(tywN34^X!R!Fc$R?RF-WX&C29Y$Q3y2)|pnn#+EJv{MCVjVR=q2oUdW0xBaF4W<rk50kE)Dy&jUfF+K4@Q+LARgq$rn`np-bRaiZ`l?5eq8y?t^y!l(b3Gp4FuWB{6f0E~?bDhpmN?`j$@dtM+!_=&oK@SA*ltU#Y#Qd3qEXz}ZnLVjNUs^@-EiW}q0Ng2Jz=FID5?E}D1(S<aWd^2b2TKjpXaraVVBhT?#D;Z|ir4y}LzSPbZ?bc2iQ2aQ&|r`8ix{%|GUV=CS(H3fT5hQ{#l9uksR-#(Z(1YqOQeM@tgv(8{MWs?-N{7?SdzdiZZzkK}dzy0Mm{^bAs?7!lA%Bw&B{ZD@QF1~8FU2&FGNp`Ig-mV*!@zq2zmtm=n1yvS||8#y`T+|@PPFAMb*y_wmKR0vBWC|DF5o1%X%5v57Y(Pnjqe)@C$KMS{`s!MWmcWdy0SGnS_uIlC>xKfbM@{hzZnNs}7G34@#%s4<UN04hTeE%Gg<~(;%V|0~+S2^SV(&F5FpCb8ea_UCp)&`)#idb7?g2yF3|#)AE>+oR?QX8W2ziy@&*k>U7yOQc!)b+rGiep$1~Z5<E~_liElRLd7Ur7;(70Vndy;(5N`+!D$q{f#1h@M|gw?xrivOHl>Ti5esrklIG5i4N?f?>HfmyI7ZlWixMQg-m+Zv_iw}sL?qWP4LY3o7xgM=@84Dm~8Q=@Xv$OUQpV62(9Oy7CUTt%9=6E5#SYu4%2PF);2U7{7(vc^+0(=jL6Sl6ZlbIQ3R&M8*|EazOv#!@n}WYE<aR2yN5Ai2M;3f1has`7beMODzEyQ;G0m(pYPp&8sTP27tPAAEH;UnW``jvo&~ASHksix9)k%G;`-hLtd{&`DfI{>!i5-a+I@&P<>XlV9NXUH^u36!yF7bKkPApK3lJy;!cO5%{W7W~)((>`8NmGDQaZT<0ZlO%9lQnBmvc@cMlDmAfK;%Ny%Tr)Ved&=&6Km$)K3v|y^PRXzl70Q+cnD7L-=xrxcC@f-Q`^^~~Ft*xfH5t$?^X%VPC&amSfEh~L|*4!tk*D;&26PTNxbCX*E9@1$$L-N`4KljGVUtSjht&%{Jr7#0n{otIw!*mMX{*fKBEHC*QP04M=9mI^`<kt7G6$-0?2U`l7qcng!t3+aaMORhQ#>VSbI6{L-DSo9JaE7D4OP(rs33f`~en4`BP;r)0&)+}$ao%o-IjHO-o&wr9P|GqRFc=IbfRKxma4yo1vJ=cXH9rrH#zp0|Chzn4M>!nhS|xUQtAmnK5Eo!~XUjS)Ddp60N$<Q0J8MgGID%c7Zo&zzlf5=tste2BAnAwfxo`$_5@8?@?W`CsP1JY(g5MgnDfW|NI)BV11wt46Gx5IirLFDFEna=VqnjrgTr&$^`!xx1Ku~faWgKNBNXc99ZuQ`QyZ0~M_xoRH?cP6H??2b}tNRCP-nJNrRNpZJEuW$lp4iRjw_mu@*9Tzfx=?a}?*sD1e%owc-Bk(=32-gDl;J8P-5ui6nweqnqpD8^p#43ZIBaWw?*RWfUdIPxb&d9*%;pQ-?Jo0tL=aHa<_mRzSHwvi1H>d}k_)gr(`Pik)13aX23Adrs!I_98AME5wj&PTdyDTgeVJ-VCZ|`^^k6Y}<#DR@RiC&@jGI?kdn7gz%m#Z&Fq=$+Spp(sdT7KDkj0<e7wRr;7)bLl#Vvsm$hLNY)#$oCzAm#>&PTfv_Uy`FMYUrRuJ}-`YushqmU@|JXNj}8zRp6nXsB<^Emy{#(VqRY?IOqm+>Wg=F1QX9S1ad%rlChM&?#NG8nO?#=gTb&blI_@?Y%mycm7BB^$d|Zb8Jj`@i4!qTf@}77;x}i-PvRwE#iwrjN67Kz=2(0G&tIik{z3SPq9OaE0xR!U|DRb?czvfA^fsjoHHiXA;W1%EV*?IDNTQpH`P{Z2!Ymva<ke<fbgsrXHUD~Ad}GyE-)_jZ46sPHzS=++{-aCq^g2*?su{fW1`5IV!kMecIogd<DYna)J0_>-~A3JZzdBwfk)xJZi;cf3$Np?Y<pi$SjxTJ;OJbPo;2-$;dqJ+m`LeL=tQM<r9_|aqoHY(H}=g=^i7c#scfb*t{wz>>NTODFOK*?L<$aP7a-m-xoewevW7C&?!rzjk8P?uT6V#gj#3}H)QO<8sR{!OiJ<6sREUa%El$BnB&<Wg$TG95VTwj8>qQ#_rUQ{V7i`G3dG~5I@6x{Q49aoXy751`)3n*Tst1X}iy^!l5u<Hm;5;7{e+OG|`EJ9Jy7vSiLM?o0>QVGTPWf}8=cHN6oLET*Wurc})2m$7-YJc+J%zihg>U)GI9g4U?cwZW2LPlpv||5^uiW=#P+fX#kG&yP<Pb-Q3qkuUbexUAJ#7M3Xl5IL)mt|I8q4T1?S57yMWhz4q9ROO*K_Sy6v(;-jtp4VYx6QO@?2<>uB!x@c$vrGOmE0NZ?!sRX?@*-)UfQFTwLxT3voEcxt=oW8ZWYnkNiqtk6pPuX@kL9!dYvPwoP>~fdRE4GizRpwxfd^JK9nS?^SM^j#m+w=t#htx;`y0Bes!bXy*khh;LJBe$cu7`-|->-L8)bRR;J6Uc5LOff0n_cvT7^Wq70XL{kZO$M7Fg?GXyp8B%^UJO>D2{&F-KEFJ|%ywDZk0Ul|bk0ZIew}fA>h<SQ&47rNjq8hoXVP*5tH^r*5lr3{hjvqm{;igUmwh3oas9cQKSkK7Ps~g4->LtRX4iIVUirW4rg=BkUyo<tv8zxV#feN4GD%CW_p(lG8#iva`uvwsXPOJj0a9N;s2u{B-UT~ZSTP4Qmny?sOABWf)>z|aLqtvzi-4DO{=x;v!?xP<(`pHM{ee~lGfA`Vve)z|azWwM=KYI70_xmbJ)eZPCu>hMC!Yl^TB@hV6xMAgPfqe1F6t2%CVv5tc)ukEJ6^)+r3~oz`9`t!?v@L~oQv$A;U&V_lm7L~Hn}b>9-3ClLp9CoKLf1N6a04#*OEb#YhBwZEWUVeNQQ|Z#HD$uOCb8CV69`;HqVAekE-i2f%@&oj1+_^tro=sS);Ow@RSWq!cM2+zKxzuMy5~=x84IN9Z90n@#$;9SN_CFQ$;HX`Fx*gf5F9~-D_`&!8l7uH0Yw+CDJx4RbOuI5>$4&FB5l;9o_Y|nF1f_7V^;!G9K;RmSDj!cb%Y(=?QS*!5PdLT<aqyd387R;Z4HuGq)_Xwrq*L2*WD|cT#udf6noRy7p1r2Cg&q3HroTL2XYdHbc?bH=Y$TBq^EmyGLg=Cpd~Y(rU$XyX4~Pbn(;mq@c_fY)tFL#n8Bk=O#~ifYw|z-*FSvn!#{fR)1N>2?zf-(<M)5@vv-lLJ^9wZ{^H@EfBfw~`}F-k{N#^+?~_0MCwluw4?p?Szkc$UfB(q`KY#MA?|k}`pFa67fB)(Ge}msX{@#E7_<Mi->380L@+aS*sD}Ub-A{h_7f;^**{6T-gSsT?|Kg`Ve)63^fASZ9|KuOPN1*@vcb@#gzkc%1-~03@KY8+d-+J<oKgG8n{xO~q@Mh!3AN*(z=LH@={HC|}=pSZqU(o_y(OIUO<3~S&7x|<2U@`U{fAIa(o5D9^k#QY)$ZD}@?aEg&AGCM2o0e@a1!qD=3sT1;#dNQG!Ii5`5b&N2E{Hh*lGH+E;+HmCvTJi0AjHHb9Vl#?_mL7b0RlfkIK$(cVLHfc)U$SkTU2cX+f=^Z&!=HROPI=a;3><gjrL=m-<cYbr*3d4*b)-mI4b~1_M=odrw=4RTY4CILo1qDdnHZ)7i3;NZ)OY?*p|Ahz$I#?YUZ_IQW>J{As%uEk=~7CruX8tWB>|rIx24C)#z}*{MPY&0v?I%!8A|T>6{)t75sSpnftrYRx)~UonE~3;5wL8UJ}tuoES}GPU)B;(p}u~i#hDZNMR_aIh*6j<cu!U^MAYbg&jyU`2eyzK0q}Cu8-xsix1vL3t!|-Q{+4ku*!@)Ml$H(12{Q;fHoF&4P-n+%Pzdfl;%j1Phe8vJTqK|4g>w=d;jB?fAu%N__yyo`O)`)xXSO3|MCYP|Kb0B@&`YDP+%Sg?_m7sC&&~YKl~FQVvj#~FL|I|J^F_!@Po%6{OyTHW!rvzrmgXbdZT1_nKlk+1GGUW948?$7K%i_6YM!1=^QWMV&aAY@o>Gjqr-?9908L_dGO00{L!cHf8&#PzwuxY%^~6_vKOy$^>_sE|4?ND63f1l1oXYYr8cMyilmoGPDIiBDlu_nQF6e>tw_~(*6e%|tV6!d*CpHG5fycFM`xg@XudlP87;K1UA4D%R<U|3A(f@m9WCycd20~?g_-P_;t(DWHaX3X;!u?qET&=tLUttH0F~@`M<wDWK|&H3NhaNPTSW+PW*qSOK&n;9q|KL0WaLJXmDN|?pI*L9Wqe7rX5T<jy;nldsgLM@LK>U(m*&Q|d}5OQL%I}jwj1bym@(?tcVjMRy4k&FT^nzGmk>HhTrL^QRYmY;_c8aA2879;AS=0Xpu<c<t}~Dm)K$}vS1vu*_t^JTt45M)^<d;9=daX(o>)*%>M52a+p@B&*eOpO$(QR6d#Yk44spUVT4qm|9dV-SRC0NHE*n<E1fij);ge|#GB4A-t5Qa=j+BDzozV%0jWMtQiU>e~%46-AE$@`3MV+ESZEFG|ePEHcnnAB-FEh<nG7*aA)Y2Z+a>m$Wlf)QhGEvE<rL)Fcr(2sd(>kF-GzMf!BZ?SWxw?9g?Wmpy^$fz$;Ki9Qd9vGUb;x*gvN$(-LK0R^+^Dn3Ghib(n@Y<@O;KL9(WO%&0TAoeKchr!muy0y@B779CXoYQz<w??ZIbSXe0@ooVhTS~rVjb!W9p5r7J_J6^vCLr=OsYvn)CV<=Vih3FE{5K2#Hi-=K_7DAh7aegtw{n_^u>#uGBY4Qm49)p;bg(R5?~dsTACNi}q(#b639Utoh5v^ClJbh*|=31Wr6u-IpY7>0BKh#Jje4D)UWuFw2F^?ebo6d4o*-bgg<%xpAetef?1VlJ?E3LATR%>H2mTKGvXD?o)rUYeKNENJju?Dm@IdH{n(o<-$)bx+#U$Pco*_v%Y`cL#}F0lq}g0h17yB&i3hs!xZm@`r56#Z{B|W4)RwvMQCPUE^rvqbJ!+NJ2Z)zyvp|Lf%e9HR{G}S&B|d^UNF7pzB^tQu`>BxzN~n+buqjH$ovF!u31=)Ge5}#`K5e^eBd7yddl;`Wz<g>Oi*VfcJ6e(BVJsvK9?bW(g23k>1euEDac@JClR3e5hrD_V<jYluYGoAhUUxWVyTLU%NtQgm?Jljk0E?5mk7IZ2UDCr?nJft%QhZy5;#T9ji|kM&0Em9gIRopNv=F*SeTUnSfLzK#+(F@zhH)^demMuXfDe#stal>ACarUh-0Dae)yC(T_y=r=cFtFZ#r_Vs^}mkg<38Oc14J=So&tL1!$h~+d(hP=sM74=-TUY-j_3qH7k>fs;yU}a%FpTNHG-U@|0ETDV(q{w}tDc8z!?Pe<LABf23bc_X%6eT#izc1Te4C*^j_?cZ3ZA`eu{1wu#6JNso-HPa5Xn+%DlQKw|7QJ5zV4%Caiu5avJ-i}P^brGMZyOhAb-c*-D1MQUgUMwAIGwqOf8GD7Axy2#V+j#-dVNK!X-JX_z3^YV>y2;L;4Y=+{C>v2s7G2a^_UcUhu7H%EKg9U7{7Pg7$_Ar?kbwYk{95a+OH01@|dxcnzAA%fgYMAs#DPx#cv#O1mJ3$6(C-8)HE%9+jU{hxN*<u<~`qBt#FS`lgWC1r0<ErL`MCd&M5O8!_<<c^l%3VTYCGPg|rr*s<#sGV!u>zxIMd#)S5#>Bv0*Q?aN{$6cHHfUM6TscR<0zrkXWWT~jIIulir!QLH?<v37t8V2-KHakw9j;PcAV%a81d8cLQ0^FvS5T9%G`K2^%A?sxpPt9)|5`DMf7~Qn9g%Z^hSv)QDFz-q}2kS4pnG^X52`r_3#yr*_GA+YY{qpqXWNo%=b#5Yet+#RJ#y3W%+5C+78a^N&tB#4M6-tVL>??XP`1clsrh66zwfhq<H)8A|LaKhAK|emiInq>N9X_x~GT%{-QGjLQ7Xd15GGGblpQ#(4oo|xT8pnkEw8niwT{ZF&nr5E~rBdOqu;E!GMo7YW+0JXZ=2N%qAGmKrTdWgj^xP9Z-4s6dhiBsP!&PNXF+Cj|)(gK(Ne6wFDuHGVQlYgBhIGeEm(z&y5+M4neWY(Xa?w61_v=obwPQcr63nehZPNz(Rnj<C%uZiANXtp_oQyY<LD=Ev-TIC@3k1#2rBT;WXED8-QPODv~(qtc@Q%x5uYyKv*1_?pq8>qF0cr4l;g`Z>3@(lJ6ua!ttQ$Mbq+`X4l{u@;Z;>gI3qFQkl{_xpE+Z&-PX&E=WQzr}^O^E6z_z&E;qUC>m4xglSt-<m2-F)mOqhUwZZCE64y{xv3?sZ(~+p7K!eHOaQ8+2t_x<%rs!J+@n0S6#D`gat(}Q#2EM@`N(6F2nB#oQ3kJP;;oBNG8d!4tXSn1RV)U}fvCyz(2;kqzJBwS@Ya8X%zno2J74;DI}^3xoh0#Oxcp}62DG~f>Rk+<-n!$&C+DRF=Pb=zcU1C2EP=<Rs@5xxrBQ@OlPE(qDr*R+lFBI{pP6w9s1(kKtGM_grG#0Db;OEFdCW5I@n@z?{{OOj9p$$EiWVZeFoi1FSH(CrSFF&ey3QN4wai+Ac@6EeQb8@@lN4fTr~GAQ_4F<~hi<>T0k=ahQ%tj}Qo7?x6)tWYa-?90kGN8lr^VJ;2^ZKC11UI2wdMl#^;<fK(WRSdmM!L_&VsQ>=IQKGZ4fNZlPQ8d`rO@2MV!{WY|<&yEw|FNVhSV1t)?kOB-7e7f6}K`0QWN17M0b+Sb>`UWrgMT*NnLCW>BW9!_uR;KAk?~j6h_@rOQk({Y;s#lHlojmL!sys*^oi40yLXr8+WMmQLnll0=+)X7{w>86Bf(G#$V_#3Y5%vD6m)kf3q0e^DSUH)v)6r^xsr1|NBw{d0!{9n#0bLe(N&0>Etwd>n9fP^4kBYZJoWaUm{5#S~=&@C!PpCd92%cBox14f}S!02Gpy*LBnjM-@OFu-8D0TC7W71ks+c5X}K!&r~@ASwk+ocM@WZyiHp<jtO!wrt`&wU3o}pd3uHOfYTD{vELr!*kw7*i1)l1)wu;mb$%}GF6hbDjR^a&+S;wwkU^CB)dl>H9*JKL1rMd`ACwb(!X^R?KJZFaP1LX~U!s8-yOd9pH}7I&8kRyVqxl)&Dimf_*oGoV<U`ru%wC(IRAT+3IN2XF>B2r8@0!!uhigLMO^JnocJNzIYrhmg6tV#hSS3)?(kqTwA~X-JF~vA9w>EUth!*KP$VA(;PLiX?SYL|5CxmLdRvH?{_bRr+DlakoT&5PvNfwr^<6~Jg6)8|RzjU;%%ib0am?mmr)8!C57M4&_3`Jn-U?k+{d~A|}=k`m-{j35~ik=IUtEnBn5_%#a2c)X8!h$ra5F`9GqF;^nC|4U+P*T0S49kvO>XZiAwhL{Y2tc8rn$<C1Q!6e+yk1S=0$s8dR<K_q3hX#@H`Ji0EgxFo0E(ss!nbK1lu=A=7RrnaMvvumpk1a-sCR()Hbrv68S`Nh%_NkJl2m{*g5I@+LMSPMsKshP=PzF6Il(%+bZO!sPA0A3#j8Nj=va=x#=XDUU$F&Hd;o_+z3UV;xmH@EL>xd@A)2euahT+EV+4h{^4vgGc{*zd;83{AYG@#b@`duLF#5YOfFCXf2?ir_r{DlY3>h`xC=mo@9<#~?$*Jk6oHqwslP3=OTpc&T`+cZ=j(C*J(<&J>!PGjJHGUB})~w5I>4WX9Z9pT_FPXwYbNha^j9gSTXP4!^VfKHn@KVjp4!{zXwErmt808MtU>xcRb{!9)kJAM2-qJ-~SI6MkwQK`7>M(&hkkAhHuqSl)1nQU=SfMn!GG?%54z_O1F%@cVuVN>IZ^};uLN1+C#F(!h$BHOU3TA>l?1n`eZmBO|ik*@D!2ZxEqu{;<du<4Aw!Ij)src820+8`1&g3Vc6vXVA>D5^J<zgKfOyzK&|E%}Q;}3q0iJalA1Y36G9;dG*UrN%~Zr`PIjbK8$wC>#;KYsX~8R)PEW}bPVnqPx0^XOd)ME=`Cz7rasUGoNy{uyWT@x#A@56Z!(B@3`XT&hd~wnP`YgA`#_(UL_UpcdJqAHfNa<Au>;$wIKe<A?u5XGH#DhT#iPY|bY~RR1;oclY4t;}8CB<{bkfOfb6I?1)e_?8Xe>C*E)YZR|aM_@BT)QZFowzhR{J9zXn_(8R;vw>t?01Wg`ry72onc{J@BXai&G?*v<)b2oEm{ODh%IBP$Z+ac5}2=j!o96tWwhoF)sj~~9z3+W9ICvLy&?LU6_-9ZhkPf)*a<g=JOdbhm52v&al(T^jr&BFDR7?1vQ?*L$b6MKPgA3yv%zzv9454=}6a(xbkzMsIDxfr=aK$Y<w!+8ThfAk*JdGy}AoS}-bjWLXnK_(dv1O0HJqUQlUV6{MGfiIYa08nNCk;Bkc1erom3nI0svX?sb6sG<I?>FN;%5MDtSMtvQC@$Lr?=@=Yf%nyNUtkdvYvK0|HFFPemp$;-*VjGzyR-Zszf@37DZ)5IiW^+#`n&YNJ3TH5^_j=NP0dBUD{)D^DRl$=$}q^NSEbGaZ!Q^Brc5YEiR7MzIe<y0s|+5!mk4q3f9H38N4U1(EY#?3i^Tm7ISOY*Z>!B!{f;xctViwL1j@sEtMDct{rK_2@6YyEYqoMe-QBFR2Gt|CeYn%&)!X-eE4+K_jo0tq3#P=EbV>+5>RhwwHHbKPG-rfDux%rU%SnBRjiWG(O2oWa$5nY(D4*NxUYF#^rznAL)!KG+Tir<1XbE!Jdt5der*sE#amqbV^)T1&*^~9O?%5-2k&-W~o^kTt+OAyf=3t+x>FEHM>CG&ri+U+8PQw1!)5Kw3z*Ui5V`f#1CZf2T@)}yc>Knc5ay0Yy%i0giZ#p%1iYzX8X`rJHFAIzjo*kBamyKOHg4HQ48Hgl{(lLI@M(p(9NY_`15H5>;R?ZRzm^b@Oa4x0?FcQo{O1ZX54?X3>1+TnZ3TJky)8Jz=&H#-UMKTF6KRSLd$A%C)q6turV{7V^0Li_<Wu(A+u8pBl>EsZv1Mie5XT`qjx1klM+jB(eHW8F69Z|paf*I>1*95!b_>2jlOyspn9nZe}Tp7>lUTw5WiBWy0^b=_%qh)ySawGplBMPnTz*FJWl)#g7n}}LujzFAd?O=bB?zQ~Ovi!AFCbnlxr^8mEg*;Za!pl((awCxY#!Q{!IyB>OhrxvAKsyBC16O{O<?b+ytL%AtpsEk0_p5s2107_jc+H@+?IT-rd)w#)kvY`mJ~T%_sJPFhzu0c|iU@ThsW|eMkB3VKy!RYr0Ua9&UIl&Nh>EIicp~i#u{HtLDXC=!(4EyUvHr}CLDlo)IFCd%7jqwva;4o_a&F1tr2TiFzNl%K7f)y882!_vH#)CFu5?EY*O4zdK-P>2=N=}jhDP^Ft+;;5&E|O2YvOukmgi*$xF{TrnXi3pYgrhciom0r2|!3w434uG&;^=O??$#y@-3%2ZQ&Is$bUiJ>xJZi4VBkr4_$XuDu{+~N^P;nzjCYunL{z_Ct#u9(dWbsYBMMI(jL|1YLKV_#t>!dl=GavMbe#?zrN1o(btEYdYD)<+Mo;iLJ`6N0GvK48e!nUC3lDF%|X&QJgSZPv=cmHteV9$e2tZ2y05xkNm(oFRT8}S60{$y+Blxfd;V=ql$ztoHps`B7p3^@F9nv)9kAetqrlQoJi4in?`>{`n8n~lTn7QpSCH=F8&9Q1f@P#+0dS)Rv0Xk#zb<4DvWX5cmQAjeMNLq*1VQMbOivC$^|L`wFrXPl$b9!%EUp|aaIbIUQt@dpsRIFMky$HfL<)UA%+=a<<HVCVxx6wPMtNCY=64um4B>;2GRkQ9oHpd#-?}>5*D%7X%|jGGC@!4?1<(AQ3k9TubvZ%khVtV1P{gybjz<kI-W52EvJN_;G3(7}YsMKi<1SO^r$OF&iwtXNTg+n4A*QmgSK2Zc?I3qRw4sbn2MN2Ne-L)w(#lxn6mhvfHz~0T%0QJOYBN%SuA_jla~OzqQwVk>X8Bth+fWts(PDM5_?kzJfy@N^1g{r5R8)+N5;$TVEec~%=e|j#2}8;g6pBcLv9ozGM139EH=aENB&{oRYbqOStG)Jr0XK5{Z~'

PROBE_KIND = 'vov_longchau_public_pages'
SAMPLING_METHOD = 'targeted_host_reservoir_unique_urls'
PRIORITY_SHA256 = '5c49501718972e18e09e89adeb2b09d023078daaf775266719a7c4064c35b087'
PRIORITY_PAYLOAD = 'c-qZ<O_SS3mf*d91y&s!0A-T(A-l&-iJDNCs@IBEOA4uM_eQw}fg-^oTqHpQph{&;o##Bqy=}zA+~zQcJxw3>&>Q<d{7?3MCNhBpizT@?M&WP~K<3NLmoHzwd|&zC!Gm|pWU&a=n~Oy@8LX4bG)N|sv@C;3zFa3oR_3cwa8cyfWm*LDG?}JF8Dy*AZTizDEvvF0<m)QSSIHuHmoCy(wpk7zJb17-EAl14SWPxXk*=!YY*THD6sj)Qc~J$Y^8#SK$@9g@hjg;3@}eIkWsHq3(kh*r`c?X&T4WbmZ?#!0<n5lknp8LIwA81|pR=|8{t-H_zn4ih*WY=`;MefZg8jxSQl$E;oNuaZVV*V@>mo-4nKw7)XO%5YwTcFqtuDE#O|gKMhJevB)%M`sk9oH0<Bw@NE0Rkc^<t4u5UB=T$N2`v3^;;v$!1Yavq`nL_ua|ccQ0PQ3dTXWF0#DHs+$4y?jjw$|GRMSo0DhHp#XgSxEH{`@EJ_(x(Y`@i0Ip&SIb2|S+8N@6Krh%1HSp<L-Xx&@%X3l!SHwehx-rd*Vkd+1~_<`tS&b|ISBdZ`QZ3f|L3_>Nmp)-Z`YRvkkH`8s)SiD(t(l@);)w$w|jegFveh&R@ZrP6{kg!7x7iLnnsf%xn3X+(hrl~h~Q1rD#;cw!m=tNd_7#`*J%;;=xa6$vNBthRkE6-__-eljA0?o4&iM|^ji+!y?c3rtqE-WSEN9+s~{}P1@|QU6u`&%Y6bIo3e=HrDi8Px7l<n^@kx!m1#0VV0co)XK3UyFw3Y>pv%CoCISN<FGKJNeFE$9Bhu#muODH&<7TIc+2Q#RWPVp`O3aUJyD(};<=jp<9RW@)4y;|n5*0WL&0N||dKySjyA};~7e21<C*I6}(L10sB7<X#b=tL~kiy2bgwgnPErPZ!*xbUaUQ1|RO6XR)9(8eocG6#m8t}fFUxpgEgxgT8Q(;I=UY#LVnm9B_U@SZ04st-S4s8VK_Rq3*fdbd=O%0e}Sbv8OYIQlYp7~r3CXJJk9{3?U?fmy99DxNQ*A{~}#QcUJiJ-&<ZNmOOkBK^ntVef~FvwwJc{_yD!<wFKg2Fp!Z1s7?Mu96ENA^7rS|A#~P^AzZcTI@M37U?2SrqJRal0}*7Bt(FYh=w0L3BEr3-4RgsG`I+-`DC-iy@zYl3=>e(LHvhhql<8nGrt>h8+Bo%L+A5)kxVRLHmwu;UI52bzKtB!e65O`k<)Zlq$vyr3k|U<a5300cqmT6_dsWWAY}AGIK)jO!n(r`xPwLE{ytXO-$%9{_0Cp#6(LEMu&a(n<)p~gRgcEm_)^_0(%w0&)*-gMj@(9@I7ci?M-b76M+c@jQ+W`5mn=3ZG0{GcDy43|!CkLtGFM+4r^P0<SAmlE5I9d1PG*B;R%Mq-#f&F3{5}{1YRt+Aan2kfbsD7!eL{5k^hcz?Bv|H1C><0+<=1He%V!Dm6VTQLdu{MOEwY*HliTV_TS-+xBg}Dr+GIuApfU#;Jz4Gr2nK06k4q3aplUQrtH~U9!404^1`e=NjsjBGb6}-O0jzkQ3V*g)=Ngu2a6O0D!5Z0E@g7I>&FSfzK%m1u3b-Zt4Ix6Q_gTI{Mk37)wHx-b1m;t79IYdZ4*#xiq8>;tAhJZ}t5jexz)>sc3Nx%xfV)fu8@d+m2Uk7jUA#FF+Ha5mCTUUO-(*wG@o!bYbjIoGeJBhKX8!|q5CpGei5-PrVLNVbq*$w$T&vxA1k!U*G^G7S&@xRI24ToJ7K%0*E#tZisq&;K8gAB8n3Sl2F>e_W`TD`bhabalVTK3KkU&7Vs;TAHg`m?U5aiJ<Bk0jGiY>Pcp~@6t#JPP%teMf&D5RN9Z(v_S?txRR^q`wl%`-|N4T0IWD(Zr{PO>Tnn6Qq3;nq^4Svcu?zyeCr$#jveQaE?Ox&$5}&6Ln6%#H5kK#cKhF^#C!SqP$fc;2VQ6_>D*R@LHWj9jZnD_APc^RjYk@JLtrH9*GQez?qm7w4;NB9o52hHuN{Lv#eQr+o^(s6`53;Q4a6S!KwFqJw@g{HkY$HCzGjkCTcRB@Uq<9Q1+#Hfe-mYU7Jwco;k#o8fbr8pG0SLS78`1la@l)H!ZF$G|_P`Soi5BAHy}vl+4dbXo$BS|-^FnTg0f!Nq2JsbZ4F&g*U4X@;f@xN5$dmef=M$DL_}ulvCx?muCzT%_rm-uLWXf_4pMm(&56{7GI+J+ogXc*2BJ>!z}KXg?4ktS^0sHPm-Fxfw~hMp+$>S$S2CX+-^in~SGeF%J268P;t1-8=y#26VcXju6$ud?qM$X36CWC?=bfv`ZuR1`umtyg(!=NLN>eC10eM*{X|(%$$y;+!rz!K^FE)d7t6|IDP@!DUi-sDQ#0tkYTF~J3c$d2N)ZSo1Q#)fo&HW#=*;Z&eV^W`4m48zJNEwz$dix%^J82Qa~k~6R4e-H4-G_#&o6)ea*@&d<?M>JVUP26&|{~!2`x(n07b~<uM8pF{?YI4PlO=w7`QUp-EQp=dka4ux3Y2L^_F)DZ=^(asTUv07YYib-GAyphvIZtmb@fVPVulVoNIZfK~!OlWC+ys>A1U>l~&F*ezqO*McY_qSDwg{w0a9qw#ioDi%09D{T7QK5WC&6hvDfypOj%Kg=X6UGyMwmD3s&AJ$A&zLl_$4twp60(P70&}OZp?p}EC5QTcAjTHcjG;bY_3TT`sAmacxxk#b+X+Rt_^x*qF6YxhO-I0<s3QIgFcqhN;9Tu_I|LtDql;{#aAA+6;)*J71?Jfvw>Ec$-M8JLu>L4EpZ>3R`S2wl%$){kOHiZY}ctSTQI6RPJt^Dya`uqH&MKZP*%;!Rs-DqrfC8de6{^(p{QA-ISG<2f6h<@<;9iPTP=Ze+*$gL*@ig-)V#4Cp*EyY+hac85i4i3)yfl+{f1|})XF)C|WRTsyoWs2h+a8+vo$9E|PJ+zg9a(dB0Kq800oUL(FXbT~&1wK%^Hlj&`B`Df5+P;aPyLcSXb;9R*ENLl|1jddE%WWewjqsTC$y<eX3+)q1BFY<|v^gSv{#;SpkCeO5yPcI~%u<(ib#(^`14hH(`-hf=Y}HYK`JY4WOp;1N=1oWflOS?ze)YPL&dZ2Y91Xd9E25Mgw<Prr8UCKzk+agmKIzznE@gH1Sm7lvTi|P@K#Ph2)@DKmq2X~-rf3dW!Ko#CpFT(7)H&;RGMsr;tz+H+#<~%hJybx9QK7I^^SPb-!0Z?mDr+Ce`n0MB$jESmR?Q&(Mx9DZnsUPZAemMC*wkFXbCJNd?U7UfRWYjd0Bvtby9Y1!&7Ru}_Ji4jk1CdCFs--oN5$_nyM$7#K)fAZ{`{Z?T}viaz6A05;-*T=h}uEbGyvD;`(#NQqCpODS`y5Dutat^KIq*sGYVv^KZao2Tjyo=fese%vYaPJU;TFM>v#w3CM)9^r#boZ;Ol<y<<alx3=V`DJTaf21iw8v*y~zzF3jy5Z`8GV{}x<u2S)T+EB^uvLRWj;+|9SNgz#e|{IR2#J(eYLDoY|ny#(&MG02`{6}z@9w9lfKt$>^Cz#zzduNoR_S}po|bO2|9M+Zmvzeo7Lzr+81%{p}l6EBq+c;=QgG=LD8Cb9h15~%<=s{x$%wg}*I7Oy4nk0EkmN6fEsc+@uG*_u-Lz&||fr#|{>-MyWN(<tAO-ET8dHSvs+&rxH|K7wadrzxB^GJg~N5mZ5>Dj@|nZBi`#<D_IDp{UV>l<%x>VMR+;4)1Xa7NiptT4PFsfSw|mRKgjxoV`j=4$uQJ(+5MLCDJ9n8!6r2pEU^4d)q~m8vS6;B(v(+ys`z84i<<LQkUP`Su&q7iG38(iY;?YW|=Kn57Q_d`u4#t!Gvn6R5A-`{tX~Fsz>4D+D22ii?rzL3e1JuI@k?Ad78KIQ~-<GABFqVCRsoS@pZ2+q(2U&H5C2!eEarnkzB^g03vR{E$Om$9AQIav_=0Qy0=9J!isdDQ*YBcI=Z8k0YQnOZXnGA;^6_xkbCx~Y)X?sudNUn?d>FytBxXOtx0UOCZSFLq_*GlExp~w7H)t=u4#c8a^x}2!jJ89?(Hb}2rq6!XXGwq*fq0q?p4P&j;Q0#<0+|r@o*}-OaXHZ%OQ=?CCZiejON-SSq`!I1EqRL=gzN0>t3eVu?=ZbXPZT=;%Y4nkq(1l10|IC29&_4V!umsjz?{<&*B%5tZ#y1!!96Vdj;*%(T5KxAv`t)&v`nzTEp%NTL4VdO&}^^qeADdP!CkbMqCWVc?EGzFS&^IUhODacZ{HLpF)l)#Q}-oG65S644<$EHtMX5XV5j<j>U_xHkU`B=aUf%9CeLIRzQhaiS{tqzh)rO;qGkE;IuWAHM>lTEAr#O{_O{$VlQ@k#^afyO}e@vRUx+>19TD*3dI2)ZFuhB{n=T`<N;U|#rS#cJ0#^iMuM=OYJgHd+$vtV+d~?zYxPUslNwP;8SUdt$y~+ZA>`GB6==r=%kh61Hx7V0WIX})d<=5i+`t9`yT|k~>Lh?TSLroP2y@T8xEWf^Tb2v;NC&!r0Y(u|PK^f;5f=GH6h0KX*^OMpm7v_#k0M2j2Ta#h2I3-O5=}a1=f?cXlV(j{-LxD<YwFfT+Q}~V8p0jP^X*BciRBKd6l()(`l2*MPGih!(*(BXP=sugEP(;+o^l<2gB08&GI2W4cWpoSCXyG=_rHDnGFWCMxSmetHcd49BvLu`!IA)wwb;ela-BzP5_TjKwbP7!0-&SvL62qaUvF0Mq^6v;=7{FtBBwim)8MKWxzsR_ae^~Uf@&c!*C{jFc_IK3lP_puVX>94?saltnb;FTtd#T}ISe_fyMUUUi;&dhzz)-7={7Fb2L3ceC(Ed5TOiYsJ;vHu?P<#pVfzr+M_eQf8;N`x4$tp}ig{Ab_si_(w2LMi$2OaETIXf5vN=wNpHDr%3Er?<63FUcA5~S1TK2%fG=CiAaO5klaiB&fSfs#Q%U}vjc|lGHGVj(gj~pk#5y_YdBlZ~?HBY0K4xUQ%iCKG2^D{c1W{W9uO)Ba?ueq&K4HpgR!rLqw>HQ{0Dj?IM3#vnv1TNv1sE2V@Bh|laF*;}Cq(IXnz~C+G-T-7}Yi~N%wX!a5u35<Sob?|Ty`*o*r^yN$Jiz|f;0N0AJGg<XvXoZu!yOtpLr$S^y0}Hupa$U#teR@Dk{T9@MD-*pH~CEbZs#v|DxrDWn`&o;zD@PqUW!Bde@oo(uXC!}4<cM~vfQ_7X1A<o_ha*P?m3Ay1@t?GEB7pBxzDk2LJwX7h}S2OtpxBiUxROO3Wk?C`fqsHi0lCCd>xUW9hx$Gdw*L)#iuA=?48Q!oms?jWw<@h$U^qQiYYXwh-a_|qJLZ18t(4OoQDt9yNV_67?rbQ`#iPRrLLCXBELh{HZA#Hx0TSMEUrI$&?LBhAFZ`1*IBS~ez~W;sB;nATM<#8Y`SF)Jvoxgo1Xj3v}CA8k+Tt-Oz@2eFleT6XP{G`!l%$ax8`U`_}Irv5+FhGT7azptHGMAH671Ld9p7Ak2)wZtHQ|+oC&6+0+>{6ab4xB!7Qm@V;8~;&<z=mf2bZTjiy6YY2eCm?VL1f1$1rbAP)rOA@u=-?3gQ%_gqV~YJe1|W%%Yb#{e;Y2X+Erw3mA7;OLb47-DzLIh|<V?6R<R<BDvYaO@Ivm}&J>Rn1n(+t34Coi#)&tX;SJrQI@}_8rM+;EWUa&(?wic1=y0xR_>)E1jL+1IeLkxUClUEMco|38O-!PG}-M+CfMtYv06F*rs6e$J*N?=beDGzi*2C5|_9k-8T@mQmOh4&b%14xCC7?@X5susNy$ESRg1DtfeLiCrY`50L0A&iVSH@y|`xL7!;ya9D~F6V%8TgQ&?zFaJpIGB3<N{u;T(N8@mTt^vy#Rtoh#4ImXTcGRy~zWx4AhRTnmTxd1sYN~Z`gy#$V6Njuoxq3JC?SY4*)8sd2Cuw0NPJ{ILld?8quCzvGI$#FX9-GN(~i8#eoC-J0FqsK~=yO-qI>nsM&5m)H{r@^p8uy0;Lp?ZRuI&Nr!t?8pdM5NT5ZkFp3ES!ubR7~2)4Gl{4i+Qolb`=_2Yc&h`649t%_-Q*dK^RnF?`j+0TP@UK6Xy<y=EvPYF;DTqp)b8#TuCMDEpbv#vW%VB(PqEclyl20Qct5xS1y|C$rPu1UaR%)w&PaBOb2&Hwve|sOIKfG2&|!haM#Hf<KVFUUGaIwKh8UfHPL_8V&Aqp4eUWz5K95J`#Zq#F0IBS0C&f(&};j2BTm|f6q_Zqx^TZ`MaE(Ds5dha?&8!c(ljdgbxiUGjR@J){y?hOU0|{Yv#CeIeP0)yac0;F6z=9WEJy0-Qw(8I+6C|hq$L3Ewqs`-lwpBt(#*ZU>6-2!6;$W%2G9~DM&05CQ`qhRTd-^}_u)4xl)9_<zqU3_*xg9hZ0w5T1~cziBWByNwj#E@HaDVetft-1r)_?@Lk~Ev8)d6IEvH~HMW;-3W%{Zwe&>fn<5E_RA30G?ds|$~qoV_;_~lo_p2j?l55LA&5QG+kEoqFRU*8|_wAU3cQpTJPq%^hL=3N-mN)Db<3Q>zzN1_G4OoxX=%UWyaFd#YaeN(=Bk8&MpN66?G3EFB@Z8xxe&&6%0z&c*NDf*Se`pAM>J3%oi+K#X@*iBo<+oza$qw>V$B%ts_3an@-+<&C;dH@d_5o+7}FWClo01`Z+-RE}bi;lz^K-BO>w}a&1^k6z&zxRfZ%cv)F?0j8Tv~_&c{DGWfe?VjB*1^>AQkMJ;I=*a5M`1d;h!34LkTzy%z0d$SV(Lk7u&vqWRLFxhY;=`gCJoHV!f_6hlwN=+RK7OpR4YAkLX6s-(Q@4f5QK(pYL7J3uvVx2f!{<DQG(1v;=2t0&>yJ9irgiQ@WJZSYx@nrT1AI@*aRdc4eJ&m?|>?o>*ykMg@{pC9L3}o5yzf=E0ogPs_UB?3`h!w&YZwHPXlVm8U-dYFSH{TQS|v+#4Ls`PN5kmjt)~Gt!CL}S5wR{yId(e&1r6t5qljdXF{DY$LNksz^g`vm33$skU>Zkf~N_8U{kohfs8EVFmh84g%Y3|q{*l-QdvIeqw)4o-M3~gVvNb~)pbVQ1{`xnZR+0v1hQbI)osh`jlc*9Mj*4WtEWe|YJW@L`le%&OMo4f$UWm8t{FS4Pi@`E7N?AzT`OEy2KAcIC)T*%)2+fkH^;`MYnxX^+tpD$zZ~`OHg0X<*35<cIvYv(b4@J4s;sfA6pQQ#XcPscde4$-xCfWuz_+GU6m?3kh^$YZxfdwbdD@?)z`<>2ngT5)C$l;{fctcYz|VbHgS;_oN1bzmr$iHdK9h*{_($+#B3Rb+{w8>h+u?+EEEyS?`GH<8VbG{!mf9oo-WHKXCI=WHOg<oXLy@oZOnki2!<#ZiSSGkyLw4X8$!cs;*V>{y3=XZ%cp&$Xr6**9(mDKk83*n@Y&B$Y_t6oyEw{4fIF0b#9;5l>s#|P=A9fY!P`QX2xz-k$LscE8x>?0|0$`U^6rtrfz+0Y`@(*k<eNXB@60P)DB!9_Dj~Q>QP<ly02>ca43?Y{v_&G;4&GwlxIN1#SG}KZWlk$0YJ}0ckjbJ@Hona(x`y5H=3wlrh6R>1x@k#zYgvWJ+1pFpY1W=MfOwB>TNF{Yd@bH)yEqKkq&Oi&O)X;G(i;K7c^c2%iNEC?X^H_o$%;#g?iSs33j#gmpZcDR@?v-Zsx{zso(c;>IG{I9RnkIN8Su{Ynf&JciUqilf%6F`EXI+v-Yg2XYqb;x3*4DTa7_$$biz+~^^=gYYI#9NL8pF$50G|e|5D~4PP8glhR)d5xG{Sq!t_XShoJ^grdbaJB=rI-!DFnL236E5ub!>lhy31B{L?dH7%I2#wnSlxj5(vUXcnpdr^go;3>G#++lvx$-V=I#myPlW#f_^`9XO}<be{zc_n-s0if1*`Ut$r>hnwn8<RZHXSC@Hrm$5f}_AFX36<a<7{YUtX#!EynQQIr#vLI$K*R;;^1rSKiA*`5d!PJ#Jf6=*EXB{E7tQKW&U4i2i)7z(uTRGaZ3K~tTa6B1x7C}Y|zD=<S^GMUG+?*TuL)zyG<sVx@ac4Z7BHTeD8*WbT8c^jXeygQBm^!oXUtqH-dU<eQPqJ>#G-dv!+1?mq9=7(^SC5~E9E*2i+tzZ&2$DI#G<^)46a7rw%8>+e-d&HrZcWQ%rZ06X_4!vI9_hXLU&d@ZP^h5%meuKe$ntli=BjSPGc{3$A!NgL0luJH^rsy5iLb3prX&oVPzI-PL!%wk*kUIg^WSznTq9vdK<mA$dB!-^_Mp9+g7Wh2pctto?Zz!`e)Y;FdoC*FdS>D80QO>s|O@*Z-Kjb}4_rkiGLvI)?f>Wm*8+FTpQUvOgr46NuGQ#?2p?m@&w0>{Rqxr}5Mt)&<wLghwgE~0-9;TDy7MMQxZ6iMt5{6x9nSwHBDxnCkI0M8`F#<lHDV*LoGC>ml@n(RIcwFbXB6oCG00+aw6Sf7(r1foe(=J8x`(nyeu0%o9Xi{lM3aPdk(pv4E9h_sWx}c#i{M4djCIuyBec=!@np926v%%r{nXdKop6c<Kpo|KyA?Se|2O@RqZXdhL44xfb19?4>)$NV05m5Vk$@cCB*7~;4+S#(j2T^PTSG9CNa7S1G%-zYzn;Tm3&`ddEx20v8A=I5+;-s+21t52Hu}D@|p$bwB;nLW5M_tT*mAuE?9hg(*Iw>&IVcS)<F_OB5YG7;Cm<QZ>Cr(&`GCT_gep>wX-&XTry1Dt~FaJ~pSM$IAFVIvie)*4oy$Sx$KmXr<|K%_LG6`0{{N;aSPzJtS{Pllk5Dov!fBf%i=;oL<-hGJ922GL{8&BHc)mp%+SFXcZIzeqkFX8NhN6<88bw4GrLC_y*nns4#4(S<m9v}%?Zz+K0@7?EN1{rIMQz#gWfW>QUxCJ0E?&}hjuNhkU<~<6Wjil99f&xxT?UJjT30Sn@jdZ;p=W~)XcS;USJ;&0Vzc#qZH{{wm-#}QVDdBJyvL)h6Oe=Tz)j>NxzcFc|wy(8NJWCBF)I(H8)(PSJft`S}nVy6kPD?#7;>jPjv^oowxzK@O8C8X+?sa-0L{o0oSZTP)wEFky#dG@hJ#=MnhwO3wY?hjaQrir-CAgGvs>#cfe6e9S8<NOIPCI_Lavfe;@$#{;9N7=8D#T`NUgawR@Gc`IHdKUIrCLIW1-PTPB5gEh&YKf%8|$*}U*Nn@zJ4_M#5KsN;4p$pp{TcQ91dkfO3yaM_8=)PT+tD=IoF+I^$(KSBB!N=2>(p8>o~c?T7mnL?ncHd8Z?$$oxMv-(1R7np<+ynwN9q%c(R;=jY`C{KVZ>Q2;E>Q>5SzedT}lf_#>M-+X|1(79h?k<FH5g-b!-yK34Cw=ay`~*NBc3U^FWd*a_h=aKr$qteOqJ4(|vlF>Co#LrQ+d=mmGuf2A~67=L3%Y!3Y<G^Zn$Ciy6Q>vp(aUDl(OkzvgqX643hb$JE4$A}+Fak3$$W1L?#{I5XrQ7#6t3syHGT;mq(o`3B$LyCwyz>|fVonn3JJ8B*W<dg^QNlY$Y`ul`9opwyh_T(yNgtB-f*$1i_{9?S5*`%H7R^hR*OWC{d*OrLfkGI~4+dH%W@PxZcHgw1Ok(H_23{Q4`p9ic}>^zQGDSsZctWMsD2vJrkCR~Nv`KU7kR->Sv&U|frR)QBJga7Br?qVe$?as%-!AxGI<frJ?>$L59El0OrYwH21^Ptyt+G)9z-I>t9$p_f1{ngG{5g==zI%Z0OaCdQ=J)JxldsZwACfx%wFXd3xd($LfF~FERIAgF*b{~w81w88&SeA{(iGktvF_jtVkTh46V@%X}!2<1BG2t9~e5i}<lv4t$ZCk%qvmOWbsON{sC-EWqaYRM<57FO`I_?GfM=~<v=?czFg<=&h6xzil-Rc~l7O7M9OINSZv}oXFGkoD6fach!VRdKNkkupEs4*tSnpPz#C-zRTX7-jVy=X~^iab+mAo99TQNVhv2fxbN!CAv*;kvDYX~{!DA1y;m@Q7T^FtQLHj_5)5j8SlSOJi<NeeF$u;PLFEA=q=`vm^OD0zrr<Dz+r;Om}TC13XgSRZVb~&1~$GBPY|WnZ>+lAXP~d9KqcW!h=1!ukW6{j-P+~`k3>_nq#X#s(H3Hx*dXZ{{rm@_T+}g-+cQlKK|y}e-B;TN2jd<RcPi;1vV4@@tfDL{#$Gs?Ri^e)(OHi3Y{^l0@0-d_X5Yay=I*+(BT;Fq7*Y^a7>jzc%;iS;P%7<q6mRG;O-5#4qha9Gu%s1Dt|E7WCVEeN@-{MYWa*qk5J!$UaK7*iWCI`{D$`FW?Fm<V&Q(K7)(96LQ?=H#X7&WO{F$G$2zpY{<>Q%_NSIA@uHO#XT31}+aZI13GinWtAim!@3qa@_?oYLGkDc|B5Ae3>!i9G91ahLWj;KB5k~S3UJVYa-+oECkLx#fIE&W4C^B#Z8635N$fu(gEc-(f<f<$m0U!l~kHF#j@@-vw>3oD|{OUlsyjWsS4z(0K9~uxJt}aBoWp~!RJ!nDuGaeVw2Rn6uelV|V*ngfTXqj@$@w>w|+*$SkOb$@?kts{B>f!zwZv!f@$dM^RuZ~*Kz}5_+q*Ih%9a)Iy7fpmu(C^8vh%c;a^w_Ml29H@tVw)9ti+bEogBqi^^(LiG2!<|guL5Na#xSSGIWA2}FigefU3;DcuY7gv@c8-5myv*IB<JCCgIt%;1|S{M14s*otT!pfqpSzv8S9?S6ZAX#Yq7dUAhm{+8PvJGOz?h>30QC`6%E~GJA^ynK;#OMi=qcOrXOv6pems^TQ7W8OA3|ha^Zc;yhz*z{^YWCGv>#E^kd&N8*Je805UQPf8d4>UL8nqeD9$Qve)!ZOkst%bfNB!WHU@R*)$>x<$)(4YctJT*uj^f_kssv@bGop{Vwj<Woz>dzfRy+<dh?cGW^2fXu3f6Vr<8kMyDPymH_MwR*f{@a0?zRwAGT6Ig^$gP$6&o!Sj>fzj*d4{=?hXuTD>1Js-oxNGpsZDcD075dQKwe)jU^ID{>J&(-MS2@`xaWh0CraSxr=O5@V%^XgU?j`{{=bo`8DAYTL)loEUyAwbP4`R2u&6M7G)5c~bR)90_hJ>C7b6%-#;|Ajh8n2~X)Y9%`1kYa?d_}m>VV0CQayWhONr9FS_0V|ugh)2^DnF#a?@7e>sE6Br?ekx4Dc6Z6Y6$<%@5I(Sw5Sa@@+lcPr!h+x{^qLZn4aU4k<MP?N2BbXmH&o{Cz8IVpfGnG+C^0!tAY?M=>=T&R;n+1gZ;+%-j%UR3Le{w;;~4z(6Kd-qxUs=BM0{(rjmDPZWvp6Gpg3F4nN&XaW5qDH_HsBQ28iJ^be~q(SNP|h%i_LQ4Rx-ft*hg{t+W<X_ab?8(2*t#1R658kq?C4E?k7i`hEmA1xG);`=STX4NhHp>B|3A7q3Z}gNn(&*|h9D4apTeI%H25I1Ga3C+toYru3!KfjoELIMNO=@W-`mTV1bm+MkswYf?j6#bxD{bF8IM=BQt!v;_LLvDtbSn>^BLBR|8zV3iN#?Eue&kP&7@vA+)$>UqD`j5JR}o-@`tc?$7E>-40O5?{D-zD2e&{L5R>mj`AG_rm=f@-o4pb&JH((UfCVYM%kgei`nnyk={<tGqTBwC|#|?){2qOyiCJS)kmbs>aBtcx9t1w%r){hjw(01eVLrdI5$ah$h!y^V$wS#_;#<0P|h+7i~ue^=UBv4qJU881|5^W1gg1_Z>fS&&1PWTtgf(8NTK(LfTSp2u`H2Zt@YVL=1IBVG{06yAHP(g*0PG!h>o0eo)SrgAF+-Zm$Ekzk>P&s$6U?!TaKJu+G+Z)nV)Z0JBAo50Ab+=>P8Ep!Vmgr?IB-E=X8*z*COPu6ausddFG17-HPe_@krnZqUHNN?`pL`G>k|4neg+@cUTrgc8pPr;>}9+D!xs0&QQS#XL;mZ4o*J(WhjRtdon344C0%4zeGJd5Cc|I*?<hW2gr{1oFFHX<EzvXW_e(`1slJHz)C%XQ$s7{|zL2F&N_#9V)HHaulqSym}#?zWCD#ND<7vKu8PaHdH3-u~IZ8<bsygD9IR$vWUP!+$8&UNgQjGiQ}tlxb&6S0a}<eIf^CI9KStzc6t)OdL4iN;??ul-`}=KVQff&6`SgB%AN46wnzpumcM<SoU#1%co2QMM^MZAL%ng?DAI6{y_?lqqL;GUj}D4W$vT=}S(MTp9!zYb$#+t+F<EXH{&q(}u@mN?UxiA1CvFoQ6UVmxND0YG9omS+$am|}!dVkTFi9KngVp%+%?b|fSE_VqdrQb5EH>YLi;r|3PWyHU9PgG5*GpK>i|le<HFvm&N@zxD{vG=gs;-<g%^KfVY7jFH9o6RO0dK60sAj0`j7sw+2FpXm<g4M|fX%jPchc+QMoz%Sx1;}r%x+H!(7=rQ$v(e+g5F6_WpE@BJ=v#Mq0eE3FM2`_-vAfxEY_JSd|CH2V>2ZZ&~UqqWZ$V0wkFc7xH@T-NO=CscX#w!`1F44dD7Qmr;N~O(mLJg?ZEfEc8}sd9c?YaEnW50`PKWq6R9xH9-}4$!?6ROaLWhD#wG_O#bTk}uiFY;HN@@p?G#pG=a2G>#&-IsPm+;chs*IHzVRfVI`XA<YUgsobhogsKu%de&_cezTQLKbQVlQTOI#lZEG%#u+!Ez+JeL#N3@?*lv8{$bO$chzKQ;33m?Te5I?#6bn?^^pl0tQ71ho=-l&zRy6J6On8o8;!J41%IXQQwz%3HHh=v^(%JQKXZ{VIDzBmcqpjx=SiD#)Qx=cz%gZJFv$E~;*PAOF8FmwSk89&VVr7e4RAA3BpirW{wcb8bG17@VoUBUi~i?t-5KRp*ow*Q1fu3qjCh5YzHRhc)Y=ixj#X1}zuO-2x;!De4|-P={f911#LY0}mR^?VhbWjKx<&F0i6oitET<@{vJr628rgVj@fLUBku)#N#e#@kBf-ZNCQBueqeDE*Ef7CuZn~B-Pjvh=?Y}WZQ*_jX5jFf);~7j@ud|W0Ox?;r&hAFYlo?yZ_w|_sZDfsoP50;wfWI5~s)WOCtUR8f|9t{{X@q#^e'


import base64
import hashlib
import importlib.util
import zlib

def sha256(body):
    return hashlib.sha256(body).hexdigest()

sample_bytes = ''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in SAMPLES).encode('utf-8')
crawler_bytes = zlib.decompress(base64.b85decode(CRAWLER_PAYLOAD))
if sha256(crawler_bytes) != CRAWLER_SHA256:
    raise ValueError('Snapshot crawler bị hỏng.')
if (len(SAMPLES) != EXPECTED_URL_COUNT or len({r['domain'] for r in SAMPLES}) != EXPECTED_DOMAIN_COUNT
        or len({r['url'] for r in SAMPLES}) != len(SAMPLES) or len({r['id'] for r in SAMPLES}) != len(SAMPLES)):
    raise ValueError('Danh sách mẫu thiếu hoặc trùng ID/URL.')
if (set(r['domain'] for r in SAMPLES) != set(TARGET_DOMAINS)
        or any(sum(r['domain'] == domain for r in SAMPLES) != SAMPLES_PER_DOMAIN for domain in TARGET_DOMAINS)):
    raise ValueError('Mẫu phải có đúng 20 URL cho mỗi domain VOV và Long Châu.')
manifest_path = RUN_DIR / 'manifest.json'
sample_path = RUN_DIR / 'sample.jsonl'
crawler_path = RUN_DIR / 'crawler_snapshot.py'
if manifest_path.exists():
    saved = json.loads(manifest_path.read_text(encoding='utf-8'))
    if (saved.get('probe_kind') != PROBE_KIND
            or saved.get('sample_sha256') != sha256(sample_bytes)
            or saved.get('source_sha256') != SOURCE_SHA256):
        raise ValueError('Run cũ không khớp mẫu của notebook này.')
    if not sample_path.exists() or sample_path.read_bytes() != sample_bytes:
        raise ValueError('Danh sách mẫu của run cũ đã thay đổi.')
    if not crawler_path.exists() or sha256(crawler_path.read_bytes()) != CRAWLER_SHA256:
        raise ValueError('Snapshot khác run cũ; hãy tạo run mới.')
    if json.loads((RUN_DIR / 'source_profile.json').read_text(encoding='utf-8')) != SOURCE_PROFILE:
        raise ValueError('Source profile của run cũ đã thay đổi.')
else:
    sample_path.write_bytes(sample_bytes)
    crawler_path.write_bytes(crawler_bytes)
    (RUN_DIR / 'source_profile.json').write_text(
        json.dumps(SOURCE_PROFILE, ensure_ascii=False, indent=2), encoding='utf-8')
    manifest_path.write_text(json.dumps({
        'created_at': datetime.now(timezone.utc).isoformat(),
        'probe_kind': PROBE_KIND, 'dataset': 'AIGuruTinix/ViBioMIR',
        'source': SOURCE_URL, 'source_file': SOURCE_FILE, 'source_revision': SOURCE_REVISION,
        'source_sha256': SOURCE_SHA256, 'source_rows': SOURCE_PROFILE['rows'],
        'sampling': SAMPLING_METHOD, 'seed': SAMPLE_SEED, 'selected': len(SAMPLES),
        'target_domains': list(TARGET_DOMAINS), 'samples_per_domain': SAMPLES_PER_DOMAIN,
        'selected_domains': EXPECTED_DOMAIN_COUNT,
        'baseline_archive': BASELINE_ARCHIVE, 'baseline_archive_sha256': BASELINE_ARCHIVE_SHA256,
        'sample_sha256': sha256(sample_bytes),
    }, ensure_ascii=False, indent=2), encoding='utf-8')

spec = importlib.util.spec_from_file_location('domain_probe_crawler', crawler_path)
crawler = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = crawler
spec.loader.exec_module(crawler)
priority_path = RUN_DIR / 'priority_probe_snapshot.py'
priority_bytes = zlib.decompress(base64.b85decode(PRIORITY_PAYLOAD))
if sha256(priority_bytes) != PRIORITY_SHA256:
    raise ValueError('Priority probe snapshot bị hỏng.')
if priority_path.exists() and sha256(priority_path.read_bytes()) != PRIORITY_SHA256:
    raise ValueError('Priority probe khác run cũ; tạo run mới.')
priority_path.write_bytes(priority_bytes)
priority_spec = importlib.util.spec_from_file_location('domain_priority_probe', priority_path)
priority_probe = importlib.util.module_from_spec(priority_spec)
sys.modules[priority_spec.name] = priority_probe
priority_spec.loader.exec_module(priority_probe)
if not isinstance(USER_AGENT, str) or not USER_AGENT.strip() or any(c in USER_AGENT for c in ('\r', '\n')):
    raise ValueError('USER_AGENT phải là chuỗi header hợp lệ.')
crawler.UA = USER_AGENT
crawler.ROBOT_AGENT = USER_AGENT.split('/', 1)[0]
(RUN_DIR / 'baseline_domains.json').write_text(json.dumps(BASELINE_DOMAINS, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Đã nạp {len(SAMPLES)} URL mẫu và crawler {crawler.VERSION}.')


## 4. Chuẩn bị bảng kết quả và phân loại lỗi

Output riêng: `vao_duoc.csv`, `khong_vao_duoc.csv`, `chua_chay.csv`,
`loi_theo_loai.csv`, `can_xem_noi_dung.csv`; bảng tổng là `ket_qua.csv`.
Các CSV dùng UTF-8 BOM để mở được tiếng Việt trong Excel.
`error_detail` giữ nguyên lỗi kỹ thuật; `reason` diễn giải tiếng Việt.
Mở `ket_qua.html` để lọc theo domain hoặc trạng thái.
`uu_tien_fix_domain.csv/html` xếp hostname còn lỗi theo số URL dataset giảm dần.
`tong_hop_domain.csv` gồm cả hostname đã phục hồi; `WEB_ERRORS.md` là báo cáo lỗi chi tiết.


In [ ]:
import csv
import html
import shutil
from collections import Counter
from IPython.display import HTML, FileLink, display

STATUS_REASONS = {
    'robots_unknown': 'Không xác định được chính sách robots.txt; crawler dừng trước khi tải trang.',
    'robots_denied': 'robots.txt trả HTTP 401/403; crawler dừng theo chính sách robots.',
    'robots_disallowed': 'robots.txt không cho phép crawler tải URL này.',
    'http_error': 'Trang đích trả HTTP ngoài khoảng 200–299.',
    'network_error': 'Không tải được trang do lỗi kết nối sau retry.',
    'unsafe_address': 'Địa chỉ đích không phải IP public được crawler cho phép.',
    'invalid_url': 'URL không hợp lệ hoặc dùng cổng ngoài 80/443.',
    'oversized': 'Phản hồi vượt giới hạn kích thước của crawler.',
    'response_deadline': 'Tải phản hồi vượt giới hạn thời gian tổng.',
    'rate_limited_deferred': 'Server yêu cầu chờ Retry-After quá lâu; crawler hoãn URL.',
    'redirect_limit': 'Vượt số lần redirect cho phép.',
    'unsupported_content_type': 'Đã tải phản hồi nhưng định dạng không được hỗ trợ.',
    'unexpected_error': 'Lỗi ngoài dự kiến; xem error_detail.',
}

def classify_error(status, detail, http_status):
    lower = detail.lower()
    if status == 'http_error':
        return f'page_http_{http_status}'
    if status == 'robots_disallowed':
        return 'robots_disallowed'
    if status.startswith('robots_') and 'robots http ' in lower:
        return 'robots_http_' + lower.split('robots http ', 1)[1].split()[0]
    if 'html instead of robots.txt' in lower:
        return 'robots_html_instead_of_txt'
    prefix = 'robots_' if status.startswith('robots_') else ''
    if 'waiting for domain lock' in lower:
        return prefix + 'domain_lock_budget'
    if 'request budget exhausted' in lower or 'retry budget exhausted' in lower:
        return prefix + 'budget_exhausted'
    if 'winerror 10013' in lower or 'network is unreachable' in lower:
        return prefix + 'environment_network_blocked'
    if any(s in lower for s in ('timed out', 'timeout')):
        return prefix + 'timeout'
    if any(s in lower for s in ('sslerror', 'certificate verify failed', 'ssl:')):
        return prefix + 'ssl_error'
    if any(s in lower for s in ('nameresolutionerror', 'getaddrinfo failed', 'name or service not known', 'failed to resolve')):
        return prefix + 'dns_error'
    if any(s in lower for s in ('connection refused', 'failed to establish a new connection', 'connection reset')):
        return prefix + 'connection_error'
    return status

def result_rows():
    rows = []
    for sample in sorted(SAMPLES, key=lambda s: (-SOURCE_PROFILE['host_counts'][s['domain']], s['domain'], s['sample_index'])):
        record_path = RUN_DIR / 'records' / f"{sample['id']}.json"
        record = json.loads(record_path.read_text(encoding='utf-8')) if record_path.exists() else {}
        if record and (record.get('id'), record.get('url')) != (sample['id'], sample['url']):
            raise ValueError('Checkpoint không khớp ID/URL nguồn.')
        status = record.get('status', 'pending')
        page_http = record.get('http_status')
        robots = record.get('robots', {})
        # Robots failure may happen before fetch() attaches its evidence to the record.
        if status.startswith('robots_') and not robots:
            from urllib.parse import urlsplit
            target = record.get('requests', [{}])[-1].get('url', sample['url']) if record.get('requests') else sample['url']
            if record.get('redirects'):
                target = record['redirects'][-1]['to']
            p = urlsplit(target)
            robot_file = RUN_DIR / 'robots' / (sha256(f'{p.scheme}://{p.netloc}'.encode()) + '.json')
            if robot_file.exists():
                robots = json.loads(robot_file.read_text(encoding='utf-8'))
        detail = record.get('error', '')
        browser_file = RUN_DIR / 'browser_records' / f"{sample['id']}.json"
        browser = json.loads(browser_file.read_text(encoding='utf-8')) if browser_file.exists() else {}
        if browser and (browser.get('id'), browser.get('url')) != (sample['id'], sample['url']):
            raise ValueError('Browser checkpoint không khớp ID/URL.')
        requests_access = ('CHUA_CHAY' if status == 'pending' else
                  'VAO_DUOC' if page_http is not None and 200 <= page_http < 300 else 'KHONG_VAO_DUOC')
        if DIRECT_PUBLIC_PROBE:
            access, effective_engine, effective = priority_probe.access_outcome(record, browser)
            if record.get('challenge_detected'):
                requests_access = 'KHONG_VAO_DUOC'
        else:
            access, effective_engine, effective = requests_access, 'requests', record
        flags = record.get('quality_flags', [])
        if status == 'pending':
            reason, error_type, stage = 'Chưa chạy URL này.', '', 'pending'
        elif status == 'extracted':
            stage = 'done'
            error_type = ('qc_' + record['quality_status'] if record.get('quality_status') in ('review', 'rejected') else '')
            reason = ('Tải được và nội dung đạt QC tự động.' if record.get('quality_status') == 'accepted'
                      else 'Tải được, nhưng nội dung cần xem lại hoặc bị QC loại: ' + ', '.join(flags))
        else:
            stage = ('robots' if status.startswith('robots_') else 'extraction' if access == 'VAO_DUOC' else 'fetch')
            error_type = record.get('error_kind') or classify_error(status, detail, page_http)
            reason = ('Trang trả HTML xác minh/JavaScript thay vì nội dung bài viết.'
                      if record.get('challenge_detected') else STATUS_REASONS.get(status, status))
        requests_reason = reason
        if effective_engine == 'selenium':
            stage = 'done_browser'
            reason = 'Selenium cải thiện tải/trích xuất so với Requests. QC: ' + effective.get('quality_status', 'unknown')
        rows.append({
            'domain': sample['domain'], 'dataset_url_count': SOURCE_PROFILE['host_counts'][sample['domain']],
            'sample_id': sample['id'], 'sample_url': sample['url'], 'access_result': access,
            'requests_access_result': requests_access, 'effective_engine': effective_engine,
            'effective_http_status': effective.get('http_status'),
            'effective_quality_status': effective.get('quality_status', ''),
            'effective_char_count': effective.get('char_count'),
            'effective_quality_flags': '; '.join(effective.get('quality_flags', [])),
            'http_status': page_http, 'robots_http_status': robots.get('status'),
            'robots_attempt_count': robots.get('attempt_count'),
            'robots_cache_hit': robots.get('cache_hit'),
            'robots_fetch_id': robots.get('fetch_id', ''),
            'robots_error_kind': robots.get('error_kind', ''),
            'robots_elapsed_seconds': robots.get('elapsed_seconds'),
            'robots_lock_wait_seconds': robots.get('lock_wait_seconds'),
            'crawl_status': status, 'failure_stage': stage, 'error_type': error_type,
            'reason': reason, 'requests_reason': requests_reason,
            'error_detail': detail, 'quality_status': record.get('quality_status', ''),
            'quality_flags': '; '.join(flags), 'char_count': record.get('char_count'),
            'title': record.get('title', ''), 'final_url': record.get('final_url', ''),
            'elapsed_seconds': record.get('elapsed_seconds'),
            'reused_from': record.get('reused_from', ''),
            'request_attempt_count': len(record.get('requests', [])),
            'retryable': record.get('retryable'), 'retry_deferred': record.get('retry_deferred', ''),
            'page_attempted': bool(record.get('requests')),
            'browser_status': browser.get('status', ''),
            'browser_http_status': browser.get('http_status'),
            'browser_quality_status': browser.get('quality_status', ''),
            'browser_char_count': browser.get('char_count'),
            'browser_final_url': browser.get('final_url', ''),
            'browser_error': browser.get('error', ''),
            'browser_attempt_count': browser.get('attempt_count'),
            'sample_role': sample['sample_role'], 'sample_index': sample['sample_index'],
        })
    return rows

def domain_rows(rows):
    result = []
    for domain in sorted(BASELINE_DOMAINS, key=lambda h: (-SOURCE_PROFILE['host_counts'][h], h)):
        selected = [r for r in rows if r['domain'] == domain]
        failed = [r for r in selected if r['access_result'] == 'KHONG_VAO_DUOC']
        content = [r for r in selected if r['access_result'] == 'VAO_DUOC' and r['effective_quality_status'] != 'accepted']
        pending = sum(r['access_result'] == 'CHUA_CHAY' for r in selected)
        errors = Counter(r['error_type'] for r in failed)
        errors.update('qc_' + (r['effective_quality_status'] or 'unknown') for r in content)
        result.append({
            'domain': domain, 'dataset_url_count': SOURCE_PROFILE['host_counts'][domain],
            'baseline_error_type': BASELINE_DOMAINS[domain]['error_type'],
            'selected_urls': len(selected), 'additional_urls': sum(r['sample_role'] == 'additional' for r in selected),
            'missing_additional_urls': BASELINE_DOMAINS[domain]['shortfall'],
            'completed_urls': len(selected) - pending, 'pending_urls': pending,
            'page_not_attempted': sum(not r['page_attempted'] and r['access_result'] != 'CHUA_CHAY' for r in selected),
            'accessible_urls': sum(r['access_result'] == 'VAO_DUOC' for r in selected),
            'access_failures': len(failed), 'content_issues': len(content),
            'total_problem_urls': len(failed) + len(content),
            'additional_problem_urls': sum(r['sample_role'] == 'additional' for r in failed + content),
            'error_types': '; '.join(f'{kind}: {n}' for kind, n in sorted(errors.items())),
            'test_status': 'CHUA_CHAY' if pending == len(selected) else 'CHUA_HOAN_TAT' if pending else 'CON_LOI' if failed or content else 'DA_PHUC_HOI',
        })
    return result

def write_csv(name, rows, fields):
    temporary = RUN_DIR / (name + '.tmp')
    with temporary.open('w', encoding='utf-8-sig', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(RUN_DIR / name)

def save_outputs(show=True):
    rows = result_rows()
    fields = list(rows[0])
    counts = Counter(r['access_result'] for r in rows)
    summary = {'selected': len(rows), 'vao_duoc': counts['VAO_DUOC'],
               'khong_vao_duoc': counts['KHONG_VAO_DUOC'], 'chua_chay': counts['CHUA_CHAY'],
               'crawl_status_counts': dict(Counter(r['crawl_status'] for r in rows)),
               'quality_status_counts': dict(Counter(r['quality_status'] for r in rows if r['quality_status'])),
               'effective_quality_status_counts': dict(Counter(r['effective_quality_status'] for r in rows if r['effective_quality_status'])),
               'created_at': datetime.now(timezone.utc).isoformat()}
    domains = domain_rows(rows)
    priority = [dict(priority_rank=i, **r) for i, r in enumerate((r for r in domains if r['total_problem_urls']), 1)]
    summary.update(hostnames=len(domains), domains_with_errors=len(priority),
                   priority_order='dataset_url_count descending', user_agent=USER_AGENT,
                   page_not_attempted=sum(not r['page_attempted'] and r['access_result'] != 'CHUA_CHAY' for r in rows),
                   additional_urls=sum(r['sample_role'] == 'additional' for r in rows))
    summary.update(request_mode='direct_public_page_probe' if DIRECT_PUBLIC_PROBE else 'robots_enforced',
                   requests_http_2xx=sum(r['requests_access_result'] == 'VAO_DUOC' for r in rows),
                   browser_recovered=sum(r['access_result'] == 'VAO_DUOC' and r['effective_engine'] == 'selenium' for r in rows),
                   browser_access_recovered=sum(r['access_result'] == 'VAO_DUOC' and r['effective_engine'] == 'selenium' and r['requests_access_result'] != 'VAO_DUOC' for r in rows),
                   browser_content_improved=sum(r['access_result'] == 'VAO_DUOC' and r['effective_engine'] == 'selenium' and r['requests_access_result'] == 'VAO_DUOC' for r in rows),
                   browser_tested=sum(bool(r['browser_status']) for r in rows),
                   browser_http_2xx=sum(r['browser_http_status'] is not None and 200 <= r['browser_http_status'] < 300 for r in rows))
    write_csv('tong_hop_domain.csv', domains, list(domains[0]))
    write_csv('uu_tien_fix_domain.csv', priority, ['priority_rank', *domains[0]])
    (RUN_DIR / 'uu_tien_fix_domain.json').write_text(json.dumps(priority, ensure_ascii=False, indent=2), encoding='utf-8')
    write_csv('ket_qua.csv', rows, fields)
    write_csv('requests_vs_selenium.csv', rows, fields)
    for value, filename in [('VAO_DUOC', 'vao_duoc.csv'), ('KHONG_VAO_DUOC', 'khong_vao_duoc.csv'), ('CHUA_CHAY', 'chua_chay.csv')]:
        write_csv(filename, [r for r in rows if r['access_result'] == value], fields)
    write_csv('can_xem_noi_dung.csv', [r for r in rows if r['access_result'] == 'VAO_DUOC' and r['effective_quality_status'] != 'accepted'], fields)
    failures = Counter(r['error_type'] for r in rows if r['access_result'] == 'KHONG_VAO_DUOC')
    errors = [{'error_type': kind, 'domain_count': len({r['domain'] for r in rows if r['access_result'] == 'KHONG_VAO_DUOC' and r['error_type'] == kind}),
               'url_count': n,
               'domains': '; '.join(sorted({r['domain'] for r in rows if r['access_result'] == 'KHONG_VAO_DUOC' and r['error_type'] == kind}))}
              for kind, n in failures.most_common()]
    write_csv('loi_theo_loai.csv', errors, ['error_type', 'domain_count', 'url_count', 'domains'])
    (RUN_DIR / 'ket_qua.json').write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding='utf-8')
    (RUN_DIR / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    md = ['# Các website còn lỗi sau lượt test bổ sung', '',
          f"Thời điểm UTC: {summary['created_at']}. Run: `{RUN_DIR.name}`.", '',
          f"Đã chạy {len(rows)-counts['CHUA_CHAY']}/{len(rows)} URL thuộc {len(domains)} hostname. "
          f"Tải được sau Requests/Selenium: {counts['VAO_DUOC']}; chưa tải được: {counts['KHONG_VAO_DUOC']}; chưa chạy: {counts['CHUA_CHAY']}.", '',
          f"Requests tải được: {summary['requests_http_2xx']}; chọn kết quả Selenium: {summary['browser_recovered']} "
          f"(phục hồi truy cập: {summary['browser_access_recovered']}; cải thiện nội dung: {summary['browser_content_improved']}). "
          'Các cột lỗi Requests vẫn giữ nguyên dù Selenium phục hồi thành công.', '',
          f"{summary['page_not_attempted']} URL đã xử lý nhưng chưa request bài viết vì dừng trước bước tải bài. "
          'Các trường hợp này được phân biệt với bài viết thực sự trả HTTP lỗi.', '',
          f'HTTP User-Agent: `{USER_AGENT}`.', '',
          f"Bộ mẫu có {len(rows)} URL thuộc {len(domains)} hostname, gồm {summary['additional_urls']} URL bổ sung. "
          'Gồm lỗi truy cập và lỗi nội dung/QC. Số URL dataset là số dòng của hostname, chưa loại URL trùng.', '',
          'Ưu tiên theo số URL dataset giảm dần; không suy ra toàn bộ URL của domain đều bị lỗi. '
          'Dừng ở robots là chưa request bài viết, không đồng nghĩa website không mở được trong trình duyệt.', '',
          '| Ưu tiên | Hostname | Số URL dataset | Mẫu đã chạy | Lỗi truy cập | Lỗi nội dung | Loại lỗi |',
          '|---:|---|---:|---:|---:|---:|---|']
    for item in priority:
        md.append(f"| {item['priority_rank']} | {item['domain']} | {item['dataset_url_count']:,} | {item['completed_urls']}/{item['selected_urls']} | {item['access_failures']} | {item['content_issues']} | {item['error_types']} |")
    md += ['', '## Domain đã phục hồi trong các mẫu', '']
    md += [f"- {r['domain']}: {r['accessible_urls']}/{r['selected_urls']} URL tải được và đạt QC." for r in domains if r['test_status'] == 'DA_PHUC_HOI'] or ['Chưa xác định domain phục hồi hoàn toàn.']
    md += ['', '## Domain thiếu URL bổ sung', '']
    md += [f"- {r['domain']}: lấy được {r['additional_urls']}/4 URL mới; dataset có {r['dataset_url_count']} dòng." for r in domains if r['missing_additional_urls']]
    md += ['', '## Chi tiết lỗi theo domain', '']
    for item in priority:
        md += [f"### {item['priority_rank']}. {item['domain']}", '',
               f"{item['dataset_url_count']:,} URL nguồn; {item['page_not_attempted']} mẫu đã xử lý nhưng chưa request bài viết. "
               f"Lỗi baseline: `{item['baseline_error_type']}`.", '']
        for row in rows:
            if row['domain'] != item['domain'] or row['access_result'] == 'CHUA_CHAY' or (row['access_result'] == 'VAO_DUOC' and row['effective_quality_status'] == 'accepted'):
                continue
            detail = (row['error_detail'] or row['quality_flags']).replace('\n', ' ').replace('`', "'")
            md += [f"- ID {row['sample_id']} ({row['sample_role']}): [URL](<{row['sample_url']}>) — `{row['error_type']}`. "
                   f"HTTP trang: {row['http_status']}; HTTP robots: {row['robots_http_status']}. {row['reason']} Lỗi gốc: `{detail}`."]
        md.append('')
    (RUN_DIR / 'WEB_ERRORS.md').write_text('\n'.join(md), encoding='utf-8')

    visible_fields = ['domain', 'dataset_url_count', 'sample_id', 'sample_url', 'sample_role', 'access_result',
                      'http_status', 'robots_http_status', 'robots_attempt_count', 'robots_cache_hit', 'error_type', 'reason', 'error_detail',
                      'quality_status', 'quality_flags', 'char_count']
    labels = ['Domain', 'Số URL nguồn', 'ID mẫu', 'URL mẫu', 'Mẫu cũ/mới', 'Vào được?', 'HTTP trang',
              'HTTP robots', 'Lần thử robots', 'Cache hit robots', 'Loại lỗi', 'Lý do', 'Lỗi gốc', 'QC', 'Cờ QC', 'Số ký tự']
    if DIRECT_PUBLIC_PROBE:
        visible_fields = ['domain', 'dataset_url_count', 'sample_id', 'sample_url', 'http_status',
                          'error_type', 'error_detail', 'quality_status', 'char_count',
                          'browser_status', 'browser_http_status', 'browser_quality_status', 'browser_char_count', 'browser_error',
                          'access_result', 'effective_engine']
        labels = ['Domain', 'Số URL nguồn', 'ID', 'URL', 'HTTP Requests', 'Lỗi Requests', 'Chi tiết',
                  'QC Requests', 'Ký tự Requests', 'Selenium', 'HTTP Selenium', 'QC Selenium', 'Ký tự Selenium', 'Lỗi Selenium',
                  'Kết quả cuối', 'Engine tải được']
    def esc(value):
        return html.escape(str(value if value is not None else ''), quote=True)
    body = []
    for row in rows:
        cells = []
        for field in visible_fields:
            value = row[field]
            text = (f'<a href="{esc(value)}" target="_blank" rel="noopener noreferrer">{esc(value)}</a>'
                    if field == 'sample_url' else esc(value))
            cells.append('<td>' + text + '</td>')
        body.append(f'<tr data-access="{row["access_result"]}">' + ''.join(cells) + '</tr>')
    style = """<style>body{font:14px/1.5 system-ui,sans-serif;margin:20px;color:#172b4d}
    .table-wrap{overflow:auto;max-height:70vh;border:1px solid #ccd4df}table{border-collapse:collapse;width:100%}
    th,td{padding:9px;border-bottom:1px solid #dce2e9;text-align:left;vertical-align:top}th{background:#e8eef6;position:sticky;top:0}
    td{min-width:95px;max-width:330px;overflow-wrap:anywhere}td:nth-child(4),td:nth-child(9),td:nth-child(10){min-width:260px}
    input,select{padding:8px;margin:0 10px 14px 0}a{color:#1654a0}</style>"""
    table = '<div class="table-wrap"><table><thead><tr>' + ''.join('<th>' + esc(label) + '</th>' for label in labels)
    table += '</tr></thead><tbody>' + ''.join(body) + '</tbody></table></div>'
    heading = f'<h1>Kết quả thử crawl {len(rows)} URL / {len(domains)} hostname</h1><p>Vào được: {counts["VAO_DUOC"]} · Không vào được: {counts["KHONG_VAO_DUOC"]} · Chưa chạy: {counts["CHUA_CHAY"]}</p>'
    heading += '<p>Vào được tính cả Requests và Selenium phục hồi; trang challenge được loại. Các cột Requests vẫn giữ lỗi gốc, QC là kết quả riêng. Kết quả chỉ áp dụng cho các URL mẫu.</p>'
    controls = '<input id="search" placeholder="Tìm domain, URL hoặc lỗi" aria-label="Tìm kiếm"><select id="access" aria-label="Kết quả"><option value="">Tất cả</option><option>VAO_DUOC</option><option>KHONG_VAO_DUOC</option><option>CHUA_CHAY</option></select><span id="count"></span>'
    priority_fields = ['priority_rank', 'domain', 'dataset_url_count', 'selected_urls', 'completed_urls', 'pending_urls', 'access_failures', 'content_issues', 'error_types']
    priority_labels = ['Ưu tiên', 'Domain', 'Số URL dataset', 'Mẫu', 'Đã chạy', 'Chưa chạy', 'Lỗi truy cập', 'Lỗi nội dung', 'Loại lỗi']
    priority_heading = '<h2>Domain còn lỗi — số URL dataset giảm dần</h2><p>Gồm lỗi truy cập và lỗi nội dung/QC. Số URL nguồn không phải số URL đã xác minh lỗi.</p>'
    priority_table = '<div class="table-wrap"><table id="priority"><thead><tr>' + ''.join('<th>'+esc(label)+'</th>' for label in priority_labels)
    priority_table += '</tr></thead><tbody>' + ''.join('<tr>' + ''.join('<td>'+esc(r[f])+'</td>' for f in priority_fields) + '</tr>' for r in priority) + '</tbody></table></div>'
    (RUN_DIR / 'uu_tien_fix_domain.html').write_text('<!doctype html><html lang="vi"><meta charset="utf-8"><title>Ưu tiên domain lỗi</title>'+style+priority_heading+priority_table+'</html>', encoding='utf-8')
    javascript = """<script>const rs=[...document.querySelectorAll('tbody tr')],s=document.getElementById('search'),a=document.getElementById('access');
    function filter(){let n=0;for(const r of rs){const ok=r.textContent.toLowerCase().includes(s.value.toLowerCase())&&(!a.value||r.dataset.access===a.value);r.hidden=!ok;n+=ok?1:0}document.getElementById('count').textContent=n+' hostname'}
    s.addEventListener('input',filter);a.addEventListener('change',filter);filter();</script>"""
    javascript = javascript.replace("document.querySelectorAll('tbody tr')", "document.querySelectorAll('tr[data-access]')").replace("n+' hostname'", "n+' URL'")
    (RUN_DIR / 'ket_qua.html').write_text('<!doctype html><html lang="vi"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Kết quả crawl domain</title>' + style + heading + priority_heading + priority_table + '<h2>Từng URL mẫu</h2>' + controls + table + javascript + '</html>', encoding='utf-8')
    # Archive outside the run directory to avoid including the archive itself.
    zip_path = Path(shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name))
    if show:
        print(json.dumps(summary, ensure_ascii=False, indent=2))
        display(HTML(style + priority_heading + priority_table))
        display(HTML(style + heading + table))
        print('File kết quả:', RUN_DIR)
        print('ZIP:', zip_path)
    return rows, summary, zip_path

# Create reports before networking, so partial runs have explicit CHUA_CHAY rows.
rows, summary, zip_path = save_outputs(show=False)
print('Đã chuẩn bị báo cáo và phân loại lỗi.')


## 5. Tải trang và đối chiếu Selenium

Chạy Requests trước, từng URL tuần tự trong một domain; các domain chạy song song.
Mỗi URL có timeout connect/read và ngân sách riêng, không tiêu hao ngân sách vì các mẫu cùng domain chờ nhau.
Ngân sách được kiểm tra giữa các bước; DNS và thử nhiều IP có thể làm thời gian thực vượt mức đặt.
Theo `BROWSER_TEST_MODE`, Selenium thử lại URL Requests lỗi hoặc HTTP 200 nhưng text rỗng/QC lỗi;
có thể đặt `off` để chỉ chạy Requests. Nếu bật Selenium, Chrome được kiểm tra trước lượt Requests.
Trên Linux, notebook tự cài thư viện Chrome còn thiếu bằng apt (gồm ATK); Selenium Manager tìm/tải
Chrome và driver. Preflight thất bại sẽ dừng ô crawl, lưu `browser_setup_error.json`,
`browser_runtime.json`, `browser_preflight.json` và log Chrome để chẩn đoán.
Requests và Selenium có kết quả riêng; DOM trống/challenge không được ghi thành tải được bài.
Requests retry tối đa 3 lần; lỗi mạng được đóng pool kết nối cũ trước lần thử tiếp theo.
Selenium dùng User-Agent thực và retry timeout tối đa 2 lần; 403/challenge không được retry liên tục.
`access_result` tính cả trường hợp Selenium phục hồi, còn các cột lỗi Requests vẫn giữ nguyên.
Kết quả cuối chọn engine có QC tốt hơn; kết quả tương đương giữ Requests.
Parser 120ask lấy câu hỏi/câu trả lời; cnkang lấy thân bài, bỏ menu/bài liên quan.
`summary.json` tách số URL phục hồi truy cập và số URL cải thiện nội dung nhờ Selenium.
Để thử lại riêng nhóm lỗi, đặt `RECOVERY_RUN_DIR` là đường dẫn run cũ và giữ `RESUME_RUN_DIR = None`.
Recovery tạo run mới, kiểm tra hash raw 2xx trước khi reuse, để các checkpoint lỗi được tải lại.
Để chạy lại toàn bộ bằng cấu hình mới, giữ `RESUME_RUN_DIR = None` và Run All.

Mặc định `failed_only` tự chuyển mọi lỗi Requests/trích xuất, text rỗng và QC `review`/`rejected`
sang Selenium, kể cả PDF cần kiểm tra lại. `off` chủ động tắt đối chiếu Selenium;
robots bị chặn và URL không hợp lệ/địa chỉ không được phép vẫn được giữ nguyên.

Tiến độ và cảnh báo QC (review, domain_parser_fallback, …) được tổng hợp mỗi LOG_EVERY URL.
Lỗi HTTP/mạng, trang challenge và nội dung rỗng vẫn in ngay kèm URL và nguyên nhân.


In [ ]:
import argparse
import os
import socket
import time

# DNS check is diagnostic only; failure here does not stop the crawl.
network_path = RUN_DIR / 'network_check.json'
if not network_path.exists():
    network = {
        'checked_at': datetime.now(timezone.utc).isoformat(),
        'environment': 'kaggle' if Path('/kaggle/working').is_dir() else 'colab' if Path('/content').is_dir() else 'local',
        'python': sys.version, 'platform': sys.platform,
        'proxy_environment_present': {key: bool(os.environ.get(key)) for key in
                                      ('HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY', 'NO_PROXY')},
        'dns': [],
    }
    for domain in NETWORK_CHECK_DOMAINS:
        start = time.monotonic()
        entry = {'domain': domain}
        try:
            entry['addresses'] = sorted({r[4][0] for r in socket.getaddrinfo(domain, 443)})
        except OSError as exc:
            entry.update(exception_class=type(exc).__name__, error=str(exc))
        entry['elapsed_seconds'] = round(time.monotonic() - start, 3)
        network['dns'].append(entry)
    network_path.write_text(json.dumps(network, ensure_ascii=False, indent=2), encoding='utf-8')
print('Network check:', network_path)

try:
    if DIRECT_PUBLIC_PROBE:
        if BROWSER_TEST_MODE != 'off':
            priority_probe.browser_preflight(crawler, RUN_DIR, user_agent=BROWSER_USER_AGENT,
                                             chrome_binary=CHROME_BINARY, load_timeout=BROWSER_LOAD_TIMEOUT)
        if RECOVERY_RUN_DIR:
            recovery = priority_probe.seed_successful_requests(crawler, RUN_DIR, SAMPLES, RECOVERY_RUN_DIR)
            print('Recovery:', len(recovery['reused_ids']), 'URL reuse raw;', len(recovery['retry_ids']), 'URL thử lại.')
        records = priority_probe.crawl_requests(
            crawler, RUN_DIR, SAMPLES, workers=WORKERS, delay=DELAY,
            attempts=PAGE_ATTEMPTS, user_agent=USER_AGENT,
            timeout=(PAGE_CONNECT_TIMEOUT, PAGE_READ_TIMEOUT), budget=PAGE_REQUEST_BUDGET, max_new=MAX_NEW, log_every=LOG_EVERY)
        browser_samples = priority_probe.select_browser_samples(SAMPLES, records, BROWSER_TEST_MODE)
        if browser_samples:
            priority_probe.crawl_browser(crawler, RUN_DIR, browser_samples,
                                        user_agent=BROWSER_USER_AGENT, chrome_binary=CHROME_BINARY, delay=DELAY,
                                        load_timeout=BROWSER_LOAD_TIMEOUT, attempts=BROWSER_ATTEMPTS, log_every=LOG_EVERY)
        crawler.report(argparse.Namespace(out=str(RUN_DIR)))
    else:
        crawler.locked_crawl(argparse.Namespace(
            out=str(RUN_DIR), workers=WORKERS, delay=DELAY, attempts=ATTEMPTS, max_new=MAX_NEW,
            log_every=LOG_EVERY,
            robots_attempts=ROBOTS_ATTEMPTS, robots_connect_timeout=ROBOTS_CONNECT_TIMEOUT,
            robots_read_timeout=ROBOTS_READ_TIMEOUT, robots_retry_budget=ROBOTS_RETRY_BUDGET,
            robots_failure_ttl=ROBOTS_FAILURE_TTL,
        ))
finally:
    rows, summary, zip_path = save_outputs()


## 6. Xem lỗi riêng và tải output

`khong_vao_duoc.csv` chứa domain, URL, loại lỗi, lý do tiếng Việt và lỗi gốc.
`loi_theo_loai.csv` gom số hostname và số URL theo từng lỗi (HTTP 403, timeout,
robots trả HTML, robots cấm crawl, DNS, SSL…). File JSON trong `records/`
giữ trace request/redirect; `robots/` giữ bằng chứng robots.txt.
`network_check.json` ghi DNS của mọi hostname trong bộ mẫu để đối chiếu Kaggle/local;
`priority_probe_config.json` ghi User-Agent/header, timeout và chế độ tải trang.
`requests_vs_selenium.csv` và `browser_records/` giữ kết quả trình duyệt riêng;
`browser_setup_error.json` được ghi nếu Chrome/driver không khởi động được.
`browser_runtime.json`, `browser_preflight.json`, `chrome_dependencies.log` và `chromedriver.log`
ghi kiểm tra môi trường/cài thư viện/khởi động Chrome (log cài apt chỉ có khi cần cài thêm).
ZIP gồm kết quả, raw, ảnh chụp và snapshot để kiểm tra riêng.

Kaggle: tải ZIP trong **Output**. Colab: ô dưới hiện nút tải ZIP.
Jupyter local: dùng các liên kết file hoặc mở đường dẫn output được in ra.


In [ ]:
# Có thể chạy lại ô này để xuất báo cáo từ checkpoint mà không crawl thêm.
rows, summary, zip_path = save_outputs(show=False)
failed_rows = [r for r in rows if r['access_result'] == 'KHONG_VAO_DUOC']
print(f'Không vào được: {len(failed_rows)}/{len(rows)} URL, thuộc {len({r["domain"] for r in failed_rows})} hostname')
for row in failed_rows:
    print(f"{row['domain']} | {row['error_type']} | {row['error_detail']}")
print('\nOutput:', RUN_DIR)
for name in ['WEB_ERRORS.md', 'uu_tien_fix_domain.html', 'uu_tien_fix_domain.csv', 'tong_hop_domain.csv',
             'requests_vs_selenium.csv',
             'ket_qua.html', 'ket_qua.csv', 'vao_duoc.csv', 'khong_vao_duoc.csv',
             'loi_theo_loai.csv', 'can_xem_noi_dung.csv', 'chua_chay.csv']:
    display(FileLink(str(RUN_DIR / name)))
display(FileLink(str(zip_path)))

if 'google.colab' in sys.modules:
    import ipywidgets as widgets
    from google.colab import files
    button = widgets.Button(description='Tải ZIP kết quả')
    button.on_click(lambda _: files.download(str(zip_path)))
    display(button)
